In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import pandas as pd
import os

path = '/kaggle/input/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store'

# Check both file sizes
for f in os.listdir(path):
    size = os.path.getsize(f'{path}/{f}') / 1e9
    print(f"{f}  →  {size:.2f} GB")

# Check Kaggle RAM available
import psutil
ram = psutil.virtual_memory()
print(f"\nTotal RAM  : {ram.total/1e9:.1f} GB")
print(f"Available  : {ram.available/1e9:.1f} GB")

In [ ]:
import os
import pandas as pd

# List all files in the downloaded path
files = os.listdir(path)
print(f"Files in dataset: {files}")

# Load only the '2019-Oct.csv' file
csv_file_name = '2019-Oct.csv'
if csv_file_name in files:
    print(f"Loading '{csv_file_name}'")
    df = pd.read_csv(os.path.join(path, csv_file_name), nrows=5)
    display(df)
else:
    print(f"'{csv_file_name}' not found in the directory.")

In [ ]:
import gc 
del df
gc.collect()

In [ ]:
dfs = [(name, obj) for name, obj in globals().items() 
       if isinstance(obj, pd.DataFrame)]

if dfs:
    for name, obj in dfs:
        size = obj.memory_usage(deep=True).sum() / 1e9
        print(f"{name} → {obj.shape} → {size:.2f} GB")
else:
    print("No DataFrames in RAM")

In [ ]:
import pandas as pd

# Load full October dataset
df_full = pd.read_csv(os.path.join(path, '2019-Oct.csv'))
print(f"Full data shape : {df_full.shape}")
print(f"\nOriginal distribution:")
print((df_full['event_type'].value_counts() / len(df_full) * 100).round(2))
print(f"\nOriginal missing values:")
print(df_full.isnull().sum())

# Stratified sample of 5M rows mirroring original
df_oct = df_full.groupby('event_type', group_keys=False).apply(
    lambda x: x.sample(frac=5_000_000/len(df_full), random_state=42)
).reset_index(drop=True)

del df_full

print(f"\n--- SAMPLE (df1) ---")
print(f"Shape  : {df_oct.shape}")
print(f"Memory : {df_oct.memory_usage(deep=True).sum()/1e9:.2f} GB")
print(f"\nSample distribution:")
print((df_oct['event_type'].value_counts() / len(df_oct) * 100).round(2))
print(f"\nSample missing values:")
print(df_oct.isnull().sum())

In [ ]:
import sys

# Check all dataframes currently in RAM
for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame):
        size = obj.memory_usage(deep=True).sum() / 1e9
        print(f"{name} → {obj.shape} → {size:.2f} GB")

In [ ]:
print('     =================First 5 rows of Ocotber data set==================\n')
display(df_oct.head())

In [ ]:
print('          ======================Last 5 Row from the dataset======================')
display(df_oct.tail())

In [ ]:
display(df_oct.shape)

In [ ]:
columns = df_oct.columns
print('                     ==================All Features in the Dataset are================')
for col in columns:
    print(f' -{col}')

In [ ]:
# Print Data Type + basic Information
print("===== Data Types =====")
display(df_oct.dtypes)

print("\n=====Basic Info=====")
print(f"Data Range : {df_oct['event_time'].min()} --->  {df_oct['event_time'].max()}")
print(f"Unique users : {df_oct['user_id'].nunique():,}")
print(f"Unique Session : {df_oct['user_session'].nunique():,}")
print(f"Unique Products : {df_oct['product_id'].nunique():,}")
print(f"Unique Brands: {df_oct['brand'].nunique():,}")
print(f"Unique Category:{df_oct['category_code'].nunique():,}")

In [ ]:
display(df_oct['event_type'].value_counts())

In [ ]:
event_count = df_oct['event_type'].value_counts()
total_event = len(df_oct)
event_percentage = (event_count / total_event * 100).round(2)

print(f"Total Number of events : {total_event:,}")
print("\nPercentage of each event type:\n",event_percentage)

In [ ]:
# view to cart rate
view_to_cart_rate = (event_count['cart']/event_count['view']) * 100
print(f'View to Cart Rate is:{view_to_cart_rate:.2f}%')

In [ ]:
# cart to purchase rate
cart_to_purchase_rate = (event_count['purchase']/event_count['cart']) * 100
print(f'Cart to View Rate is:{cart_to_purchase_rate:.2f}%')

In [ ]:
# Missing Value Analysis 
import matplotlib.pyplot as plt
missing_count = df_oct.isnull().sum()
missing_pct = (missing_count / len(df_oct) * 100).round(2)

missing_df = pd.DataFrame({
    'missing_count':missing_count,
    'missing_percent': missing_pct
}).sort_values('missing_count',ascending = False)

print('====== Misssing Value======')
print(missing_df)

# Plot only columns having missing values
missing_only = missing_df[missing_df['missing_percent']>0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# Bar chart
axes[0].bar(missing_only.index, missing_only['missing_percent'],
            color=['#E24B4A', '#F5A623'], edgecolor='white')
axes[0].set_title('Missing Values by Column (%)', fontweight='bold')
axes[0].set_ylabel('Missing %')
axes[0].set_xlabel('Column')
for i, (idx, row) in enumerate(missing_only.iterrows()):
    axes[0].text(i, row['missing_percent'] + 0.3,
                 f"{row['missing_percent']}%",
                 ha='center', fontsize=10)

# Pie chart
axes[1].pie(missing_only['missing_count'],
            labels=missing_only.index,
            autopct='%1.1f%%',
            colors=['#E24B4A', '#F5A623', '#AAAAAA'],
            startangle=90)
axes[1].set_title('Share of Missing Values', fontweight='bold')

plt.suptitle('REES46 October 2019 — Missing Value Analysis',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/kaggle/working/missing_values.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Missing values chart saved.")

**A. Performing Analysis on the first csv file ( 019-Oct.csv)**

**STEPS**

Phase 1 — Load data         → 4 steps

Phase 2 — Understand data   → 5 steps  

Phase 3 — Clean data        → 5 steps

Phase 4 — Funnel analysis   → 5 charts  

Phase 5 — KPIs              → 1 summary

Phase 6 — Category analysis → 2 charts  

Phase 7 — Brand analysis    → 2 charts

Phase 8 — Price tier        → 1 chart   

Phase 9 — Time patterns     → 2 charts  

Phase 10 — User segments    → 2 charts  

Phase 11 — Cohort retention → 1 chart   

Phase 12 — Co-purchase      → 1 chart   

Phase 13 — Weekly trend     → 1 chart  

Phase 14 — Brand gap        → 1 chart   

Phase 15 — Export tables    → 4 CSVs    

In [ ]:
# ═══════════════════════════════════════════════
# STATISTICAL ANALYSIS
# ═══════════════════════════════════════════════
import matplotlib.pyplot as plt 
# 1. Basic statistics
print("=== PRICE STATISTICS ===")
print(df_oct['price'].describe().round(2))

print("\n=== EVENT TYPE DISTRIBUTION ===")
event_count = df_oct['event_type'].value_counts()
event_pct   = (event_count / len(df_oct) * 100).round(2)
event_df    = pd.DataFrame({'count': event_count, 'percent': event_pct})
print(event_df)

print("\n=== USER STATISTICS ===")
sessions_per_user = df_oct.groupby('user_id')['user_session'].nunique()
print(f"Avg sessions per user : {sessions_per_user.mean():.2f}")
print(f"Max sessions per user : {sessions_per_user.max()}")
print(f"Users with 1 session  : {(sessions_per_user==1).sum():,}")
print(f"Users with 2+ sessions: {(sessions_per_user>1).sum():,}")

print("\n=== PRODUCT STATISTICS ===")
views_per_product = df_oct[df_oct['event_type']=='view'].groupby('product_id').size()
print(f"Avg views per product : {views_per_product.mean():.2f}")
print(f"Max views per product : {views_per_product.max():,}")
print(f"Products viewed once  : {(views_per_product==1).sum():,}")

# ── CHARTS ──────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('REES46 October 2019 — Statistical Analysis',
             fontsize=14, fontweight='bold', y=1.02)

# Chart 1 — Event type bar chart
colors = ['#378ADD', '#EF9F27', '#1D9E75']
axes[0,0].bar(event_df.index, event_df['count'], color=colors, edgecolor='white')
axes[0,0].set_title('Event Count by Type', fontweight='bold')
axes[0,0].set_ylabel('Count')
for i, (idx, row) in enumerate(event_df.iterrows()):
    axes[0,0].text(i, row['count'] + 200000,
                   f"{row['percent']}%", ha='center', fontsize=10)

# Chart 2 — Event type pie
axes[0,1].pie(event_df['count'], labels=event_df.index,
              autopct='%1.1f%%', colors=colors, startangle=90)
axes[0,1].set_title('Event Type Share', fontweight='bold')

# Chart 3 — Price distribution
axes[0,2].hist(df_oct[df_oct['price'] < 1000]['price'], bins=50,
               color='#378ADD', edgecolor='white')
axes[0,2].set_title('Price Distribution (< $1000)', fontweight='bold')
axes[0,2].set_xlabel('Price ($)')
axes[0,2].set_ylabel('Count')

# Chart 4 — Price by event type boxplot
df_oct[df_oct['price'] < 500].boxplot(column='price', by='event_type',
                                  ax=axes[1,0], 
                                  boxprops=dict(color='#378ADD'),
                                  medianprops=dict(color='#E24B4A'))
axes[1,0].set_title('Price Distribution by Event Type', fontweight='bold')
axes[1,0].set_xlabel('Event Type')
axes[1,0].set_ylabel('Price ($)')
plt.sca(axes[1,0])
plt.title('Price by Event Type')

# Chart 5 — Sessions per user distribution
sessions_per_user_clipped = sessions_per_user.clip(upper=10)
axes[1,1].hist(sessions_per_user_clipped, bins=10,
               color='#1D9E75', edgecolor='white')
axes[1,1].set_title('Sessions per User (capped at 10)', fontweight='bold')
axes[1,1].set_xlabel('Number of Sessions')
axes[1,1].set_ylabel('Number of Users')

# Chart 6 — Top 10 categories by event count
top_cats = df_oct[df_oct['category_code'] != 'unknown'] \
               .groupby('category_code').size() \
               .sort_values(ascending=False).head(10)
axes[1,2].barh(top_cats.index, top_cats.values,
               color='#7F77DD', edgecolor='white')
axes[1,2].set_title('Top 10 Categories by Events', fontweight='bold')
axes[1,2].set_xlabel('Event Count')
axes[1,2].invert_yaxis()

plt.tight_layout()
plt.savefig('/kaggle/working/statistical_analysis.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Statistical analysis charts saved.")

In [ ]:
# ═══════════════════════════════════════════════
# PHASE 3 — DATA CLEANING
# ═══════════════════════════════════════════════
import gc

print(f"Rows before cleaning : {len(df_oct):,}")

# 1. Convert event_time to datetime
df_oct['event_time'] = pd.to_datetime(df_oct['event_time'], utc=True)

# 2. Remove price = 0
df_oct = df_oct[df_oct['price'] > 0].copy()

# 3. Fill nulls
df_oct['category_code'] = df_oct['category_code'].fillna('unknown')
df_oct['brand']         = df_oct['brand'].fillna('unknown')

# 4. New time columns
df_oct['date']        = df_oct['event_time'].dt.date
df_oct['hour']        = df_oct['event_time'].dt.hour
df_oct['day_of_week'] = df_oct['event_time'].dt.day_name()
df_oct['week_number'] = df_oct['event_time'].dt.isocalendar().week.astype(int)
df_oct['month_name']  = 'October'

# 5. Category levels
df_oct['category_l1'] = df_oct['category_code'].str.split('.').str[0]
df_oct['category_l2'] = df_oct['category_code'].str.split('.').str[1].fillna('unknown')

# 6. Price tier
df_oct['price_tier'] = pd.cut(
    df_oct['price'],
    bins   = [0, 50, 200, 500, 99999],
    labels = ['budget', 'mid', 'premium', 'luxury']
)

# 7. Dtype conversion — save memory
df_oct['product_id']    = df_oct['product_id'].astype('int32')
df_oct['category_id']   = df_oct['category_id'].astype('int32')
df_oct['user_id']       = df_oct['user_id'].astype('int32')
df_oct['price']         = df_oct['price'].astype('float32')
df_oct['hour']          = df_oct['hour'].astype('int8')
df_oct['week_number']   = df_oct['week_number'].astype('int8')
df_oct['event_type']    = df_oct['event_type'].astype('category')
df_oct['brand']         = df_oct['brand'].astype('category')
df_oct['category_code'] = df_oct['category_code'].astype('category')
df_oct['category_l1']   = df_oct['category_l1'].astype('category')
df_oct['category_l2']   = df_oct['category_l2'].astype('category')
df_oct['day_of_week']   = df_oct['day_of_week'].astype('category')
df_oct['price_tier']    = df_oct['price_tier'].astype('category')
df_oct['month_name']    = df_oct['month_name'].astype('category')

gc.collect()

print(f"Rows after cleaning  : {len(df_oct):,}")
print(f"Rows removed         : {4999999 - len(df_oct):,}")
print(f"Memory               : {df_oct.memory_usage(deep=True).sum()/1e9:.2f} GB")
print(f"Total columns        : {df_oct.shape[1]}")
print(f"\nNull check:")
print(df_oct.isnull().sum())
print(f"\nDtypes:")
print(df_oct.dtypes)
print("\n✅ Phase 3 complete. df_oct is clean and ready for analysis.")

In [ ]:
# ═══════════════════════════════════════════════
# PHASE 4 — FUNNEL ANALYSIS
# ═══════════════════════════════════════════════

# Step 1 — Count unique sessions per funnel stage
total_sessions    = df_oct['user_session'].nunique()
view_sessions     = df_oct[df_oct['event_type']=='view']['user_session'].nunique()
cart_sessions     = df_oct[df_oct['event_type']=='cart']['user_session'].nunique()
purchase_sessions = df_oct[df_oct['event_type']=='purchase']['user_session'].nunique()

# Step 2 — Drop-off at each step
drop_no_view     = total_sessions   - view_sessions
drop_no_cart     = view_sessions    - cart_sessions
drop_no_purchase = cart_sessions    - purchase_sessions

# Step 3 — Rates
view_rate        = view_sessions     / total_sessions * 100
cart_rate        = cart_sessions     / total_sessions * 100
purchase_rate    = purchase_sessions / total_sessions * 100
view_to_cart     = cart_sessions     / view_sessions  * 100
cart_to_purchase = purchase_sessions / cart_sessions  * 100
cart_abandonment = 100 - cart_to_purchase
overall_conv     = purchase_sessions / total_sessions * 100

print("=== FUNNEL COUNTS ===")
print(f"Total Sessions         : {total_sessions:,}")
print(f"Sessions with View     : {view_sessions:,}  ({view_rate:.1f}%)")
print(f"Sessions with Cart     : {cart_sessions:,}   ({cart_rate:.1f}%)")
print(f"Sessions with Purchase : {purchase_sessions:,}   ({purchase_rate:.1f}%)")

print(f"\n=== FUNNEL RATES ===")
print(f"View Rate              : {view_rate:.1f}%")
print(f"Cart Rate              : {cart_rate:.1f}%")
print(f"Purchase Rate          : {purchase_rate:.1f}%")
print(f"View → Cart            : {view_to_cart:.1f}%")
print(f"Cart → Purchase        : {cart_to_purchase:.1f}%")
print(f"Cart Abandonment       : {cart_abandonment:.1f}%")
print(f"Overall Conversion     : {overall_conv:.2f}%")

print(f"\n=== DROP-OFF COUNTS ===")
print(f"Left without viewing   : {drop_no_view:,}")
print(f"Viewed not carted      : {drop_no_cart:,}")
print(f"Carted not purchased   : {drop_no_purchase:,}")

# Step 4 — Funnel chart
fig, ax = plt.subplots(figsize=(10, 6))

stages = ['Sessions', 'Views', 'Carts', 'Purchases']
counts = [total_sessions, view_sessions, cart_sessions, purchase_sessions]
colors = ['#185FA5', '#378ADD', '#1D9E75', '#0F6E56']

bars = ax.barh(stages, counts, color=colors, edgecolor='white', height=0.5)

# Annotate count + percentage
for i, (count, stage) in enumerate(zip(counts, stages)):
    pct = count / total_sessions * 100
    ax.text(count + 5000, i, f'{count:,}  ({pct:.1f}%)',
            va='center', fontsize=11, fontweight='bold')

# Annotate drop-off between steps
drops = [drop_no_view, drop_no_cart, drop_no_purchase]
drop_labels = [f'-{d:,}' for d in drops]
for i, (drop, label) in enumerate(zip(drops, drop_labels)):
    ax.annotate(f'drop: {label}',
                xy=(counts[i+1], i+0.5),
                xytext=(counts[i+1] + counts[0]*0.1, i+0.5),
                fontsize=9, color='#E24B4A',
                arrowprops=dict(arrowstyle='->', color='#E24B4A'))

ax.set_xlabel('Number of Sessions')
ax.set_title('REES46 October 2019 — Conversion Funnel\nSession → View → Cart → Purchase',
             fontweight='bold', fontsize=13)
ax.invert_yaxis()
ax.set_xlim(0, total_sessions * 1.4)

plt.tight_layout()
plt.savefig('/kaggle/working/funnel_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Funnel chart saved.")

In [ ]:
# ═══════════════════════════════════════════════
# PHASE 5 — KPI SUMMARY
# ═══════════════════════════════════════════════

# Revenue calculations
total_revenue    = df_oct[df_oct['event_type']=='purchase']['price'].sum()
total_orders     = purchase_sessions
avg_order_value  = total_revenue / total_orders
revenue_per_visitor = total_revenue / total_sessions

print("=" * 45)
print("   OCTOBER 2019 — KPI SUMMARY")
print("=" * 45)
print(f"  Total Sessions        : {total_sessions:>12,}")
print(f"  Total Orders          : {total_orders:>12,}")
print(f"  Total Revenue         : ${total_revenue:>12,.2f}")
print("-" * 45)
print(f"  Conversion Rate       : {overall_conv:>11.2f}%")
print(f"  Cart Abandonment Rate : {cart_abandonment:>11.2f}%")
print(f"  View → Cart Rate      : {view_to_cart:>11.2f}%")
print(f"  Cart → Purchase Rate  : {cart_to_purchase:>11.2f}%")
print("-" * 45)
print(f"  Revenue per Visitor   : ${revenue_per_visitor:>12,.2f}")
print(f"  Avg Order Value       : ${avg_order_value:>12,.2f}")
print("=" * 45)

# KPI visual — horizontal bar scorecard
fig, ax = plt.subplots(figsize=(10, 6))

kpis   = ['Conversion Rate', 'View→Cart Rate',
          'Cart→Purchase Rate', 'Cart Abandonment']
values = [overall_conv, view_to_cart,
          cart_to_purchase, cart_abandonment]
colors = ['#1D9E75', '#378ADD', '#1D9E75', '#E24B4A']

bars = ax.barh(kpis, values, color=colors,
               edgecolor='white', height=0.4)

for i, val in enumerate(values):
    ax.text(val + 0.3, i, f'{val:.2f}%',
            va='center', fontsize=12, fontweight='bold')

ax.set_xlabel('Percentage (%)')
ax.set_title('REES46 October 2019 — KPI Dashboard',
             fontweight='bold', fontsize=13)
ax.set_xlim(0, max(values) * 1.3)
ax.invert_yaxis()

plt.tight_layout()
plt.savefig('/kaggle/working/kpi_summary.png', dpi=150, bbox_inches='tight')
plt.show()

# Revenue KPI card
print(f"\n  💰 Total Revenue      : ${total_revenue:,.2f}")
print(f"  🧾 Avg Order Value    : ${avg_order_value:,.2f}")
print(f"  📈 Revenue/Visitor    : ${revenue_per_visitor:,.2f}")
print("\n✅ Phase 5 complete.")

In [ ]:
# ═══════════════════════════════════════════════
# PHASE 6 — CATEGORY ANALYSIS CORRECTED
# ═══════════════════════════════════════════════

# Count unique sessions per category per event type
# This avoids the purchase > cart problem
cat_view = df_oct[df_oct['event_type']=='view'].groupby(
    'category_l1', observed=True)['user_session'].nunique().rename('view_sessions')

cat_cart = df_oct[df_oct['event_type']=='cart'].groupby(
    'category_l1', observed=True)['user_session'].nunique().rename('cart_sessions')

cat_purchase = df_oct[df_oct['event_type']=='purchase'].groupby(
    'category_l1', observed=True)['user_session'].nunique().rename('purchase_sessions')

cat_revenue = df_oct[df_oct['event_type']=='purchase'].groupby(
    'category_l1', observed=True)['price'].sum().rename('revenue')

# Combine
category_funnel = pd.concat(
    [cat_view, cat_cart, cat_purchase, cat_revenue], axis=1
).fillna(0)

# Calculate rates
category_funnel['view_to_cart_pct']     = (category_funnel['cart_sessions']    / category_funnel['view_sessions'].replace(0,1)    * 100).round(2)
category_funnel['cart_to_purchase_pct'] = (category_funnel['purchase_sessions']/ category_funnel['cart_sessions'].replace(0,1)    * 100).round(2)
category_funnel['cart_abandonment_pct'] = (100 - category_funnel['cart_to_purchase_pct']).round(2)
category_funnel['conversion_rate']      = (category_funnel['purchase_sessions']/ category_funnel['view_sessions'].replace(0,1)    * 100).round(2)

# Sort by view sessions
category_funnel = category_funnel.sort_values('view_sessions', ascending=False)

print("=== CATEGORY FUNNEL ===")
print(category_funnel.head(15))

# ── CHART 1 — Top 10 categories by view sessions ────
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

top10 = category_funnel.head(10)
x     = range(len(top10))
width = 0.25

axes[0].bar([i-width for i in x], top10['view_sessions'],
            width=width, label='View', color='#378ADD')
axes[0].bar([i for i in x], top10['cart_sessions'],
            width=width, label='Cart', color='#EF9F27')
axes[0].bar([i+width for i in x], top10['purchase_sessions'],
            width=width, label='Purchase', color='#1D9E75')
axes[0].set_xticks(x)
axes[0].set_xticklabels(top10.index, rotation=45, ha='right', fontsize=9)
axes[0].set_title('Top 10 Categories — Sessions per Funnel Stage',
                   fontweight='bold')
axes[0].set_ylabel('Sessions')
axes[0].legend()

# ── CHART 2 — Conversion rate by category ───────────
top10_conv = category_funnel[
    category_funnel['view_sessions'] > 100
].sort_values('conversion_rate', ascending=False).head(10)

colors = ['#1D9E75' if x > 2 else '#E24B4A'
          for x in top10_conv['conversion_rate']]

axes[1].barh(top10_conv.index,
             top10_conv['conversion_rate'],
             color=colors, edgecolor='white')
axes[1].set_title('Conversion Rate by Category (%)',
                   fontweight='bold')
axes[1].set_xlabel('Conversion Rate %')
axes[1].invert_yaxis()

for i, val in enumerate(top10_conv['conversion_rate']):
    axes[1].text(val + 0.05, i, f'{val:.2f}%',
                 va='center', fontsize=10)

plt.suptitle('REES46 October 2019 — Category Analysis',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/category_analysis.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Phase 6 complete.")

In [ ]:
# ═══════════════════════════════════════════════
# PHASE 6 — CATEGORY ANALYSIS FINAL
# ═══════════════════════════════════════════════

# Filter reliable categories only
reliable = category_funnel[
    (category_funnel['view_sessions'] >= 1000) &
    (category_funnel['cart_sessions'] >= category_funnel['purchase_sessions'])
].copy()

# Fix revenue format
reliable['revenue_formatted'] = reliable['revenue'].apply(lambda x: f'${x:,.2f}')

# Clean print
print("=== CATEGORY ANALYSIS — OCTOBER 2019 ===\n")
for idx, row in reliable.iterrows():
    print(f"  {idx.upper()}")
    print(f"    Views        : {int(row['view_sessions']):>10,}")
    print(f"    Carts        : {int(row['cart_sessions']):>10,}")
    print(f"    Purchases    : {int(row['purchase_sessions']):>10,}")
    print(f"    View→Cart    : {row['view_to_cart_pct']:>9.2f}%")
    print(f"    Cart Abandon : {row['cart_abandonment_pct']:>9.2f}%")
    print(f"    Conversion   : {row['conversion_rate']:>9.2f}%")
    print(f"    Revenue      : {row['revenue_formatted']:>15}")
    print()

# ── CHART 1 — Funnel by category ────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

x     = range(len(reliable))
width = 0.25

axes[0].bar([i-width for i in x], reliable['view_sessions'],
            width=width, label='View', color='#378ADD')
axes[0].bar([i for i in x], reliable['cart_sessions'],
            width=width, label='Cart', color='#EF9F27')
axes[0].bar([i+width for i in x], reliable['purchase_sessions'],
            width=width, label='Purchase', color='#1D9E75')
axes[0].set_xticks(x)
axes[0].set_xticklabels(reliable.index, rotation=45, ha='right', fontsize=9)
axes[0].set_title('Category Funnel — View vs Cart vs Purchase',
                   fontweight='bold')
axes[0].set_ylabel('Sessions')
axes[0].legend()

# ── CHART 2 — Cart abandonment by category ───────────
reliable_sorted = reliable.sort_values('cart_abandonment_pct', ascending=False)
colors = ['#E24B4A' if x > 20 else '#1D9E75'
          for x in reliable_sorted['cart_abandonment_pct']]

axes[1].barh(reliable_sorted.index,
             reliable_sorted['cart_abandonment_pct'],
             color=colors, edgecolor='white')
axes[1].set_title('Cart Abandonment Rate by Category (%)',
                   fontweight='bold')
axes[1].set_xlabel('Abandonment Rate %')
axes[1].invert_yaxis()

for i, val in enumerate(reliable_sorted['cart_abandonment_pct']):
    axes[1].text(val + 0.3, i, f'{val:.1f}%',
                 va='center', fontsize=10)

plt.suptitle('REES46 October 2019 — Category Analysis',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/category_analysis_again.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Phase 6 complete.")

In [ ]:
# ═══════════════════════════════════════════════
# PHASE 7 — BRAND ANALYSIS
# ═══════════════════════════════════════════════
import matplotlib.pyplot as plt
# Build brand funnel using unique sessions
brand_view = df_oct[df_oct['event_type']=='view'].groupby(
    'brand', observed=True)['user_session'].nunique().rename('view_sessions')

brand_cart = df_oct[df_oct['event_type']=='cart'].groupby(
    'brand', observed=True)['user_session'].nunique().rename('cart_sessions')

brand_purchase = df_oct[df_oct['event_type']=='purchase'].groupby(
    'brand', observed=True)['user_session'].nunique().rename('purchase_sessions')

brand_revenue = df_oct[df_oct['event_type']=='purchase'].groupby(
    'brand', observed=True)['price'].sum().rename('revenue')

# Combine
brand_funnel = pd.concat(
    [brand_view, brand_cart, brand_purchase, brand_revenue], axis=1
).fillna(0)

# Filter reliable brands — min 500 view sessions
# AND cart >= purchase to avoid negative abandonment
brand_funnel = brand_funnel[
    (brand_funnel['view_sessions'] >= 500) &
    (brand_funnel['cart_sessions'] >= brand_funnel['purchase_sessions'])
].copy()

# Calculate rates
brand_funnel['view_to_cart_pct']     = (brand_funnel['cart_sessions']    / brand_funnel['view_sessions']               * 100).round(2)
brand_funnel['cart_to_purchase_pct'] = (brand_funnel['purchase_sessions']/ brand_funnel['cart_sessions'].replace(0,1)  * 100).round(2)
brand_funnel['cart_abandonment_pct'] = (100 - brand_funnel['cart_to_purchase_pct']).round(2)
brand_funnel['conversion_rate']      = (brand_funnel['purchase_sessions']/ brand_funnel['view_sessions']               * 100).round(2)
brand_funnel['revenue_formatted']    = brand_funnel['revenue'].apply(lambda x: f'${x:,.2f}')

brand_funnel = brand_funnel.sort_values('revenue', ascending=False)

print("=== TOP 15 BRANDS BY REVENUE ===\n")
for idx, row in brand_funnel.head(15).iterrows():
    print(f"  {idx.upper()}")
    print(f"    Views      : {int(row['view_sessions']):>8,}")
    print(f"    Purchases  : {int(row['purchase_sessions']):>8,}")
    print(f"    Conversion : {row['conversion_rate']:>7.2f}%")
    print(f"    Revenue    : {row['revenue_formatted']:>15}")
    print()

# ── CHART 1 — Top 10 brands by revenue ──────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

top10_rev = brand_funnel.head(10)

axes[0].barh(top10_rev.index,
             top10_rev['revenue'],
             color='#378ADD', edgecolor='white')
axes[0].set_title('Top 10 Brands by Revenue',
                   fontweight='bold')
axes[0].set_xlabel('Revenue ($)')
axes[0].invert_yaxis()
for i, val in enumerate(top10_rev['revenue']):
    axes[0].text(val + 1000, i, f'${val:,.0f}',
                 va='center', fontsize=9)

# ── CHART 2 — Brand gap: high views low conversion ──
# Brands with many views but low conversion
# These are the brands Constructor would rerank
brand_gap = brand_funnel[
    brand_funnel['view_sessions'] >= 1000
].sort_values('conversion_rate', ascending=True).head(10)

axes[1].barh(brand_gap.index,
             brand_gap['conversion_rate'],
             color='#E24B4A', edgecolor='white')
axes[1].set_title('Brands with Lowest Conversion Rate\n(High Views, Low Purchase)',
                   fontweight='bold')
axes[1].set_xlabel('Conversion Rate %')
axes[1].invert_yaxis()
for i, val in enumerate(brand_gap['conversion_rate']):
    axes[1].text(val + 0.05, i, f'{val:.2f}%',
                 va='center', fontsize=9)

plt.suptitle('REES46 October 2019 — Brand Analysis',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/brand_analysis.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Phase 7 complete.")

In [ ]:
# ═══════════════════════════════════════════════
# PHASE 8 — PRICE TIER ANALYSIS
# ═══════════════════════════════════════════════

# Build price tier funnel using unique sessions
price_view = df_oct[df_oct['event_type']=='view'].groupby(
    'price_tier', observed=True)['user_session'].nunique().rename('view_sessions')

price_cart = df_oct[df_oct['event_type']=='cart'].groupby(
    'price_tier', observed=True)['user_session'].nunique().rename('cart_sessions')

price_purchase = df_oct[df_oct['event_type']=='purchase'].groupby(
    'price_tier', observed=True)['user_session'].nunique().rename('purchase_sessions')

price_revenue = df_oct[df_oct['event_type']=='purchase'].groupby(
    'price_tier', observed=True)['price'].sum().rename('revenue')

# Combine
price_funnel = pd.concat(
    [price_view, price_cart, price_purchase, price_revenue], axis=1
).fillna(0)

# Calculate rates
price_funnel['view_to_cart_pct']     = (price_funnel['cart_sessions']     / price_funnel['view_sessions'].replace(0,1)  * 100).round(2)
price_funnel['cart_to_purchase_pct'] = (price_funnel['purchase_sessions'] / price_funnel['cart_sessions'].replace(0,1)  * 100).round(2)
price_funnel['cart_abandonment_pct'] = (100 - price_funnel['cart_to_purchase_pct']).round(2)
price_funnel['conversion_rate']      = (price_funnel['purchase_sessions'] / price_funnel['view_sessions'].replace(0,1)  * 100).round(2)
price_funnel['revenue_formatted']    = price_funnel['revenue'].apply(lambda x: f'${x:,.2f}')

# Price range labels
price_ranges = {
    'budget'  : '$0-50',
    'mid'     : '$50-200',
    'premium' : '$200-500',
    'luxury'  : '$500+'
}

print("=== PRICE TIER ANALYSIS — OCTOBER 2019 ===\n")
for idx, row in price_funnel.iterrows():
    tier  = str(idx)
    range_label = price_ranges.get(tier, '')
    print(f"  {tier.upper()} ({range_label})")
    print(f"    Views        : {int(row['view_sessions']):>10,}")
    print(f"    Carts        : {int(row['cart_sessions']):>10,}")
    print(f"    Purchases    : {int(row['purchase_sessions']):>10,}")
    print(f"    View→Cart    : {row['view_to_cart_pct']:>9.2f}%")
    print(f"    Cart Abandon : {row['cart_abandonment_pct']:>9.2f}%")
    print(f"    Conversion   : {row['conversion_rate']:>9.2f}%")
    print(f"    Revenue      : {row['revenue_formatted']:>18}")
    print()

# ── CHART — Price tier comparison ───────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

tiers  = price_funnel.index.astype(str)
colors = ['#1D9E75', '#378ADD', '#EF9F27', '#E24B4A']

# Chart 1 — Conversion rate by tier
axes[0].bar(tiers, price_funnel['conversion_rate'],
            color=colors, edgecolor='white')
axes[0].set_title('Conversion Rate by Price Tier', fontweight='bold')
axes[0].set_ylabel('Conversion Rate %')
axes[0].set_xlabel('Price Tier')
for i, val in enumerate(price_funnel['conversion_rate']):
    axes[0].text(i, val + 0.05, f'{val:.2f}%',
                 ha='center', fontsize=10, fontweight='bold')

# Chart 2 — Cart abandonment by tier
axes[1].bar(tiers, price_funnel['cart_abandonment_pct'],
            color=colors, edgecolor='white')
axes[1].set_title('Cart Abandonment by Price Tier', fontweight='bold')
axes[1].set_ylabel('Abandonment Rate %')
axes[1].set_xlabel('Price Tier')
for i, val in enumerate(price_funnel['cart_abandonment_pct']):
    axes[1].text(i, val + 0.5, f'{val:.1f}%',
                 ha='center', fontsize=10, fontweight='bold')

# Chart 3 — Revenue by tier
axes[2].bar(tiers, price_funnel['revenue'],
            color=colors, edgecolor='white')
axes[2].set_title('Revenue by Price Tier', fontweight='bold')
axes[2].set_ylabel('Revenue ($)')
axes[2].set_xlabel('Price Tier')
for i, val in enumerate(price_funnel['revenue']):
    axes[2].text(i, val + 10000, f'${val/1e6:.1f}M',
                 ha='center', fontsize=10, fontweight='bold')

plt.suptitle('REES46 October 2019 — Price Tier Analysis',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/price_tier_analysis.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Phase 8 complete.")

In [ ]:
# Fix budget tier — resample complete sessions
# Get all budget product user_sessions from full data
print("Fixing budget tier — loading full data for complete sessions...")

df_full = pd.read_csv(os.path.join(path, '2019-Oct.csv'))

# Get all sessions that had a budget price event
budget_sessions = df_full[
    (df_full['price'] > 0) & 
    (df_full['price'] <= 50)
]['user_session'].unique()

# Get ALL events for those sessions — complete sessions
df_budget = df_full[df_full['user_session'].isin(budget_sessions)].copy()

del df_full
import gc
gc.collect()

print(f"Budget sessions found : {len(budget_sessions):,}")
print(f"Budget events loaded  : {len(df_budget):,}")
print(df_budget['event_type'].value_counts())

# Now calculate budget funnel correctly
bud_view     = df_budget[df_budget['event_type']=='view']['user_session'].nunique()
bud_cart     = df_budget[df_budget['event_type']=='cart']['user_session'].nunique()
bud_purchase = df_budget[df_budget['event_type']=='purchase']['user_session'].nunique()
bud_revenue  = df_budget[df_budget['event_type']=='purchase']['price'].sum()

print(f"\n=== BUDGET TIER — CORRECTED ===")
print(f"  BUDGET ($0-50)")
print(f"    Views        : {bud_view:>10,}")
print(f"    Carts        : {bud_cart:>10,}")
print(f"    Purchases    : {bud_purchase:>10,}")
print(f"    View→Cart    : {bud_cart/bud_view*100:>9.2f}%")
print(f"    Cart Abandon : {(1 - bud_purchase/bud_cart)*100:>9.2f}%")
print(f"    Conversion   : {bud_purchase/bud_view*100:>9.2f}%")
print(f"    Revenue      : ${bud_revenue:>18,.2f}")

del df_budget
gc.collect()

Still negative. Purchases (142,267) more than carts (101,464) even with complete sessions.
This means budget products are being purchased through "buy now" button — skipping cart entirely. This is a real ecommerce behavior, not a data error.
This is actually a valid insight:

Budget products ($0-50) → users buy directly without carting

Luxury products ($500+) → users cart first, then decide

In [ ]:
# ═══════════════════════════════════════════════
# PHASE 8 — PRICE TIER ANALYSIS COMPLETE
# ═══════════════════════════════════════════════

# Build price tier funnel using unique sessions
price_view = df_oct[df_oct['event_type']=='view'].groupby(
    'price_tier', observed=True)['user_session'].nunique().rename('view_sessions')

price_cart = df_oct[df_oct['event_type']=='cart'].groupby(
    'price_tier', observed=True)['user_session'].nunique().rename('cart_sessions')

price_purchase = df_oct[df_oct['event_type']=='purchase'].groupby(
    'price_tier', observed=True)['user_session'].nunique().rename('purchase_sessions')

price_revenue = df_oct[df_oct['event_type']=='purchase'].groupby(
    'price_tier', observed=True)['price'].sum().rename('revenue')

# Combine
price_funnel = pd.concat(
    [price_view, price_cart, price_purchase, price_revenue], axis=1
).fillna(0)

# Calculate rates
price_funnel['view_to_cart_pct']     = (price_funnel['cart_sessions']     / price_funnel['view_sessions'].replace(0,1)  * 100).round(2)
price_funnel['cart_to_purchase_pct'] = (price_funnel['purchase_sessions'] / price_funnel['cart_sessions'].replace(0,1)  * 100).round(2)
price_funnel['cart_abandonment_pct'] = (100 - price_funnel['cart_to_purchase_pct']).round(2)
price_funnel['conversion_rate']      = (price_funnel['purchase_sessions'] / price_funnel['view_sessions'].replace(0,1)  * 100).round(2)
price_funnel['revenue_formatted']    = price_funnel['revenue'].apply(lambda x: f'${x:,.2f}')

price_ranges = {
    'budget'  : '$0-50',
    'mid'     : '$50-200',
    'premium' : '$200-500',
    'luxury'  : '$500+'
}

# Reliable tiers — cart >= purchase
price_funnel_reliable = price_funnel[
    price_funnel['cart_sessions'] >= price_funnel['purchase_sessions']
].copy()

# ── PRINT ALL TIERS ──────────────────────────────────
print("=== PRICE TIER ANALYSIS — OCTOBER 2019 ===\n")

# Budget — special case from full data
print("  BUDGET ($0-50)")
print("    Views        :  2,393,011")
print("    Carts        :    101,464")
print("    Purchases    :    142,267")
print("    View→Cart    :      4.24%")
print("    Conversion   :      5.95%")
print("    Revenue      : $15,295,746.15")
print("    Cart Abandon :      N/A")
print("    Note         : Users bypass cart — Buy Now behavior")
print("                   Highest conversion of all tiers")
print()

# Mid, Premium, Luxury from reliable funnel
for idx, row in price_funnel_reliable.iterrows():
    tier        = str(idx)
    range_label = price_ranges.get(tier, '')
    print(f"  {tier.upper()} ({range_label})")
    print(f"    Views        : {int(row['view_sessions']):>10,}")
    print(f"    Carts        : {int(row['cart_sessions']):>10,}")
    print(f"    Purchases    : {int(row['purchase_sessions']):>10,}")
    print(f"    View→Cart    : {row['view_to_cart_pct']:>9.2f}%")
    print(f"    Cart Abandon : {row['cart_abandonment_pct']:>9.2f}%")
    print(f"    Conversion   : {row['conversion_rate']:>9.2f}%")
    print(f"    Revenue      : {row['revenue_formatted']:>18}")
    print()

# ── CHART ────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# All 4 tiers for chart
all_tiers      = ['budget', 'mid', 'premium', 'luxury']
conversion     = [5.95,
                  price_funnel.loc['mid',     'conversion_rate'],
                  price_funnel.loc['premium', 'conversion_rate'],
                  price_funnel.loc['luxury',  'conversion_rate']]
abandonment    = [None,
                  price_funnel.loc['mid',     'cart_abandonment_pct'],
                  price_funnel.loc['premium', 'cart_abandonment_pct'],
                  price_funnel.loc['luxury',  'cart_abandonment_pct']]
revenue_vals   = [15295746,
                  price_funnel.loc['mid',     'revenue'],
                  price_funnel.loc['premium', 'revenue'],
                  price_funnel.loc['luxury',  'revenue']]
colors         = ['#1D9E75', '#378ADD', '#EF9F27', '#E24B4A']

# Chart 1 — Conversion rate
axes[0].bar(all_tiers, conversion, color=colors, edgecolor='white')
axes[0].set_title('Conversion Rate by Price Tier', fontweight='bold')
axes[0].set_ylabel('Conversion Rate %')
axes[0].set_xlabel('Price Tier')
for i, val in enumerate(conversion):
    axes[0].text(i, val + 0.05, f'{val:.2f}%',
                 ha='center', fontsize=11, fontweight='bold')

# Chart 2 — Cart abandonment (budget = 0 as N/A)
abandon_plot = [0, abandonment[1], abandonment[2], abandonment[3]]
bars = axes[1].bar(all_tiers, abandon_plot, color=colors, edgecolor='white')
axes[1].set_title('Cart Abandonment by Price Tier\n(Budget = Buy Now, N/A)',
                   fontweight='bold')
axes[1].set_ylabel('Abandonment Rate %')
axes[1].set_xlabel('Price Tier')
axes[1].text(0, 1, 'N/A', ha='center', fontsize=11,
             fontweight='bold', color='#1D9E75')
for i, val in enumerate(abandon_plot[1:], 1):
    axes[1].text(i, val + 0.5, f'{val:.1f}%',
                 ha='center', fontsize=11, fontweight='bold')

# Chart 3 — Revenue
axes[2].bar(all_tiers, revenue_vals, color=colors, edgecolor='white')
axes[2].set_title('Revenue by Price Tier', fontweight='bold')
axes[2].set_ylabel('Revenue ($)')
axes[2].set_xlabel('Price Tier')
for i, val in enumerate(revenue_vals):
    axes[2].text(i, val + 50000, f'${val/1e6:.1f}M',
                 ha='center', fontsize=11, fontweight='bold')

plt.suptitle('REES46 October 2019 — Price Tier Analysis',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/price_tier_analysis_after_budget_issue.png',
            dpi=150, bbox_inches='tight')
plt.show()

print("=== KEY INSIGHTS ===")
print("Budget  → highest conversion 5.95% — impulse buying, Buy Now behavior")
print("Luxury  → highest revenue $15.3M — fewer buyers, very high value")
print("Premium → highest cart abandonment 21.42% — users hesitate on big purchases")
print("Mid     → most views — everyday products, price comfortable zone")
print("\n✅ Phase 8 complete.")

In [ ]:
# ═══════════════════════════════════════════════
# PHASE 9 — TIME PATTERNS
# ═══════════════════════════════════════════════

# ── Hourly analysis ──────────────────────────────────
hourly = df_oct.groupby(['hour', 'event_type'],
                         observed=True).size().unstack(fill_value=0)

# ── Day of week analysis ─────────────────────────────
day_order = ['Monday','Tuesday','Wednesday',
             'Thursday','Friday','Saturday','Sunday']
dow = df_oct.groupby(['day_of_week', 'event_type'],
                      observed=True).size().unstack(fill_value=0)
dow = dow.reindex(day_order)

print("=== HOURLY PURCHASE PATTERN ===")
print(hourly['purchase'].sort_values(ascending=False).head(5))
print(f"\nPeak purchase hour : {hourly['purchase'].idxmax()}:00")
print(f"Lowest hour        : {hourly['purchase'].idxmin()}:00")

print("\n=== DAY OF WEEK PATTERN ===")
print(dow['purchase'].sort_values(ascending=False))
print(f"\nBest day  : {dow['purchase'].idxmax()}")
print(f"Worst day : {dow['purchase'].idxmin()}")

# ── CHARTS ───────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle('REES46 October 2019 — Time Patterns',
             fontsize=14, fontweight='bold')

# Chart 1 — Purchases by hour
axes[0,0].plot(hourly.index, hourly['purchase'],
               color='#1D9E75', linewidth=2.5, marker='o', markersize=5)
axes[0,0].fill_between(hourly.index, hourly['purchase'],
                        alpha=0.15, color='#1D9E75')
axes[0,0].set_title('Purchases by Hour of Day', fontweight='bold')
axes[0,0].set_xlabel('Hour (UTC)')
axes[0,0].set_ylabel('Purchase Count')
axes[0,0].set_xticks(range(0, 24, 2))
peak_hour = hourly['purchase'].idxmax()
axes[0,0].axvline(x=peak_hour, color='#E24B4A',
                   linestyle='--', alpha=0.7,
                   label=f'Peak: {peak_hour}:00')
axes[0,0].legend()

# Chart 2 — Views, carts, purchases by hour
axes[0,1].plot(hourly.index, hourly['view'],
               color='#378ADD', linewidth=2, label='View')
axes[0,1].plot(hourly.index, hourly['cart'],
               color='#EF9F27', linewidth=2, label='Cart')
axes[0,1].plot(hourly.index, hourly['purchase'],
               color='#1D9E75', linewidth=2, label='Purchase')
axes[0,1].set_title('All Events by Hour of Day', fontweight='bold')
axes[0,1].set_xlabel('Hour (UTC)')
axes[0,1].set_ylabel('Event Count')
axes[0,1].set_xticks(range(0, 24, 2))
axes[0,1].legend()

# Chart 3 — Purchases by day of week
colors_dow = ['#E24B4A' if d in ['Saturday','Sunday']
              else '#378ADD' for d in day_order]
axes[1,0].bar(dow.index, dow['purchase'],
              color=colors_dow, edgecolor='white')
axes[1,0].set_title('Purchases by Day of Week\n(Red = Weekend)',
                     fontweight='bold')
axes[1,0].set_xlabel('Day')
axes[1,0].set_ylabel('Purchase Count')
axes[1,0].tick_params(axis='x', rotation=45)
for i, val in enumerate(dow['purchase']):
    axes[1,0].text(i, val + 50, f'{val:,}',
                   ha='center', fontsize=9, fontweight='bold')

# Chart 4 — All events by day of week
x     = range(len(day_order))
width = 0.25
axes[1,1].bar([i-width for i in x], dow['view'],
              width=width, label='View', color='#378ADD')
axes[1,1].bar([i for i in x], dow['cart'],
              width=width, label='Cart', color='#EF9F27')
axes[1,1].bar([i+width for i in x], dow['purchase'],
              width=width, label='Purchase', color='#1D9E75')
axes[1,1].set_title('All Events by Day of Week', fontweight='bold')
axes[1,1].set_xticks(x)
axes[1,1].set_xticklabels(day_order, rotation=45, ha='right')
axes[1,1].set_ylabel('Count')
axes[1,1].legend()

plt.tight_layout()
plt.savefig('/kaggle/working/time_patterns.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Phase 9 complete.")

In [ ]:
# ═══════════════════════════════════════════════
# PHASE 10 — USER SEGMENTATION
# ═══════════════════════════════════════════════

# Build user level summary
user_summary = df_oct.groupby('user_id', observed=True).agg(
    total_sessions  = ('user_session', 'nunique'),
    total_events    = ('event_type',   'count'),
    total_spend     = ('price',        lambda x: x[df_oct.loc[x.index, 'event_type'] == 'purchase'].sum()),
    total_purchases = ('event_type',   lambda x: (x == 'purchase').sum()),
    total_views     = ('event_type',   lambda x: (x == 'view').sum()),
    total_carts     = ('event_type',   lambda x: (x == 'cart').sum()),
).reset_index()

# Segment users
user_summary['segment'] = pd.cut(
    user_summary['total_sessions'],
    bins   = [0, 1, 5, 99999],
    labels = ['new', 'returning', 'loyal']
)

print("=== USER SEGMENT COUNTS ===")
print(user_summary['segment'].value_counts())

# Segment level KPIs
segment_kpis = user_summary.groupby('segment', observed=True).agg(
    users           = ('user_id',       'count'),
    avg_sessions    = ('total_sessions','mean'),
    avg_spend       = ('total_spend',   'mean'),
    avg_purchases   = ('total_purchases','mean'),
    total_revenue   = ('total_spend',   'sum'),
).round(2).reset_index()

print("\n=== SEGMENT KPIs ===")
for _, row in segment_kpis.iterrows():
    print(f"\n  {str(row['segment']).upper()}")
    print(f"    Users         : {int(row['users']):>10,}")
    print(f"    Avg Sessions  : {row['avg_sessions']:>10.2f}")
    print(f"    Avg Spend     : ${row['avg_spend']:>10.2f}")
    print(f"    Avg Purchases : {row['avg_purchases']:>10.2f}")
    print(f"    Total Revenue : ${row['total_revenue']:>12,.2f}")

# ── CHARTS ───────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('REES46 October 2019 — User Segmentation',
             fontsize=14, fontweight='bold')

colors = ['#378ADD', '#EF9F27', '#E24B4A']
segments = segment_kpis['segment'].astype(str)

# Chart 1 — User count by segment
axes[0].bar(segments, segment_kpis['users'],
            color=colors, edgecolor='white')
axes[0].set_title('Users by Segment', fontweight='bold')
axes[0].set_ylabel('User Count')
axes[0].set_xlabel('Segment')
for i, val in enumerate(segment_kpis['users']):
    axes[0].text(i, val + 1000, f'{int(val):,}',
                 ha='center', fontsize=10, fontweight='bold')

# Chart 2 — Avg spend by segment
axes[1].bar(segments, segment_kpis['avg_spend'],
            color=colors, edgecolor='white')
axes[1].set_title('Avg Spend per User by Segment',
                   fontweight='bold')
axes[1].set_ylabel('Avg Spend ($)')
axes[1].set_xlabel('Segment')
for i, val in enumerate(segment_kpis['avg_spend']):
    axes[1].text(i, val + 0.5, f'${val:.2f}',
                 ha='center', fontsize=10, fontweight='bold')

# Chart 3 — Total revenue by segment
axes[2].bar(segments, segment_kpis['total_revenue'],
            color=colors, edgecolor='white')
axes[2].set_title('Total Revenue by Segment',
                   fontweight='bold')
axes[2].set_ylabel('Revenue ($)')
axes[2].set_xlabel('Segment')
for i, val in enumerate(segment_kpis['total_revenue']):
    axes[2].text(i, val + 1000, f'${val/1e6:.1f}M',
                 ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('/kaggle/working/user_segmentation.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Phase 10 complete.")

In [ ]:
# ═══════════════════════════════════════════════
# PHASE 11 — COHORT RETENTION
# ═══════════════════════════════════════════════
import seaborn as sns 
# First purchase week per user
first_purchase = df_oct[df_oct['event_type']=='purchase'] \
    .groupby('user_id', observed=True)['week_number'] \
    .min().reset_index()
first_purchase.columns = ['user_id', 'cohort_week']

# All purchase events with week
purchases = df_oct[df_oct['event_type']=='purchase'][
    ['user_id','week_number']
].copy()

# Join to get cohort week
purchases = purchases.merge(first_purchase, on='user_id', how='left')
purchases['week_number'] = purchases['week_number'].astype(int)
purchases['cohort_week'] = purchases['cohort_week'].astype(int)
purchases['week_offset'] = purchases['week_number'] - purchases['cohort_week']

# Build cohort matrix
cohort_matrix = purchases.groupby(
    ['cohort_week','week_offset']
)['user_id'].nunique().unstack(fill_value=0)

# Convert to retention % — divide by week 0 cohort size
cohort_size   = cohort_matrix[0]
retention_pct = cohort_matrix.divide(cohort_size, axis=0) * 100

print("=== COHORT RETENTION MATRIX ===")
print(retention_pct.round(1))

# ── CHART — Cohort heatmap ───────────────────────────
fig, ax = plt.subplots(figsize=(14, 7))

sns.heatmap(
    retention_pct.round(1),
    annot    = True,
    fmt      = '.1f',
    cmap     = 'Blues',
    ax       = ax,
    linewidths = 0.5,
    cbar_kws = {'label': 'Retention %'}
)

ax.set_title('REES46 October 2019 — Cohort Retention Matrix\n(% of week 0 cohort returning)',
             fontweight='bold', fontsize=13)
ax.set_xlabel('Weeks Since First Purchase')
ax.set_ylabel('Cohort Week')

plt.tight_layout()
plt.savefig('/kaggle/working/cohort_retention.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Phase 11 complete.")

In [ ]:
# ═══════════════════════════════════════════════
# PHASE 12 — CO-PURCHASE ANALYSIS
# ═══════════════════════════════════════════════

# Get all purchase events with product and session
copurchase = df_oct[df_oct['event_type']=='purchase'][
    ['user_session','product_id','category_l1','brand','price']
].copy()

# Find sessions with multiple purchases
session_purchase_count = copurchase.groupby(
    'user_session')['product_id'].count()
multi_purchase_sessions = session_purchase_count[
    session_purchase_count >= 2
].index

print(f"Sessions with 1 purchase  : {(session_purchase_count==1).sum():,}")
print(f"Sessions with 2+ purchases: {len(multi_purchase_sessions):,}")

# Get products bought together in same session
multi = copurchase[
    copurchase['user_session'].isin(multi_purchase_sessions)
].copy()

# Self join to find pairs
pairs = multi.merge(multi, on='user_session', suffixes=('_1','_2'))

# Remove same product pairs
pairs = pairs[pairs['product_id_1'] < pairs['product_id_2']]

# Count co-purchases
copurchase_counts = pairs.groupby(
    ['category_l1_1','category_l1_2']
).size().reset_index(name='count')

copurchase_counts = copurchase_counts.sort_values(
    'count', ascending=False).head(15)

print("\n=== TOP 15 CO-PURCHASED CATEGORY PAIRS ===\n")
for _, row in copurchase_counts.iterrows():
    print(f"  {row['category_l1_1'].upper()} + {row['category_l1_2'].upper()}")
    print(f"    Co-purchased : {int(row['count']):,} times")
    print()

# ── CHART — Co-purchase bar chart instead of heatmap ─
fig, ax = plt.subplots(figsize=(12, 7))

# Clean unknown pairs
copurchase_clean = copurchase_counts[
    (copurchase_counts['category_l1_1'] != 'unknown') &
    (copurchase_counts['category_l1_2'] != 'unknown')
].head(10).copy()

copurchase_clean['pair'] = copurchase_clean['category_l1_1'].str.upper() + \
                           ' + ' + \
                           copurchase_clean['category_l1_2'].str.upper()

ax.barh(copurchase_clean['pair'],
        copurchase_clean['count'],
        color='#378ADD', edgecolor='white')
ax.set_title('REES46 October 2019 — Top Co-Purchased Category Pairs',
             fontweight='bold', fontsize=13)
ax.set_xlabel('Co-purchase Count')
ax.invert_yaxis()

for i, val in enumerate(copurchase_clean['count']):
    ax.text(val + 0.5, i, f'{int(val):,}',
            va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('/kaggle/working/copurchase_analysis.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Phase 12 complete.")

In [ ]:
# ═══════════════════════════════════════════════
# PHASE 13 — WEEKLY TREND
# ═══════════════════════════════════════════════

# Weekly aggregation
weekly = df_oct.groupby(['week_number', 'event_type'],
                         observed=True).size().unstack(fill_value=0)

# Add revenue
weekly_revenue = df_oct[df_oct['event_type']=='purchase'] \
    .groupby('week_number', observed=True)['price'].sum()

weekly['revenue'] = weekly_revenue
weekly['revenue'] = weekly['revenue'].fillna(0)

# Calculate weekly KPIs
weekly['total_events']    = weekly['view'] + weekly['cart'] + weekly['purchase']
weekly['conversion_rate'] = (weekly['purchase'] / weekly['total_events'] * 100).round(3)
weekly['cart_abandonment']= ((weekly['cart'] - weekly['purchase']) / weekly['cart'].replace(0,1) * 100).round(2)
weekly['revenue_formatted']= weekly['revenue'].apply(lambda x: f'${x:,.2f}')

# Week over week change
weekly['conv_wow_change'] = weekly['conversion_rate'].diff().round(3)
weekly['rev_wow_change']  = weekly['revenue'].diff().round(2)

print("=== WEEKLY TREND — OCTOBER 2019 ===\n")
for idx, row in weekly.iterrows():
    wow = f"+{row['conv_wow_change']:.3f}%" if row['conv_wow_change'] > 0 \
          else f"{row['conv_wow_change']:.3f}%"
    print(f"  Week {idx}")
    print(f"    Views      : {int(row['view']):>10,}")
    print(f"    Carts      : {int(row['cart']):>10,}")
    print(f"    Purchases  : {int(row['purchase']):>10,}")
    print(f"    Conv Rate  : {row['conversion_rate']:>9.3f}%")
    print(f"    WoW Change : {wow:>10}")
    print(f"    Revenue    : {row['revenue_formatted']:>18}")
    print()

# ── CHARTS ───────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(18, 10))
fig.suptitle('REES46 October 2019 — Weekly Trend',
             fontsize=14, fontweight='bold')

weeks = weekly.index.astype(str)

# Chart 1 — Weekly purchases
axes[0,0].plot(weeks, weekly['purchase'],
               color='#1D9E75', linewidth=2.5,
               marker='o', markersize=8)
axes[0,0].fill_between(weeks, weekly['purchase'],
                        alpha=0.15, color='#1D9E75')
axes[0,0].set_title('Weekly Purchases', fontweight='bold')
axes[0,0].set_ylabel('Purchase Count')
axes[0,0].set_xlabel('Week Number')
for i, val in enumerate(weekly['purchase']):
    axes[0,0].text(i, val + 100, f'{int(val):,}',
                   ha='center', fontsize=9, fontweight='bold')

# Chart 2 — Weekly conversion rate
colors_conv = ['#1D9E75' if x >= 0 else '#E24B4A'
               for x in weekly['conv_wow_change'].fillna(0)]
axes[0,1].bar(weeks, weekly['conversion_rate'],
              color='#378ADD', edgecolor='white')
axes[0,1].set_title('Weekly Conversion Rate', fontweight='bold')
axes[0,1].set_ylabel('Conversion Rate %')
axes[0,1].set_xlabel('Week Number')
for i, val in enumerate(weekly['conversion_rate']):
    axes[0,1].text(i, val + 0.001, f'{val:.3f}%',
                   ha='center', fontsize=9, fontweight='bold')

# Chart 3 — Weekly revenue
axes[1,0].bar(weeks, weekly['revenue'],
              color='#EF9F27', edgecolor='white')
axes[1,0].set_title('Weekly Revenue', fontweight='bold')
axes[1,0].set_ylabel('Revenue ($)')
axes[1,0].set_xlabel('Week Number')
for i, val in enumerate(weekly['revenue']):
    axes[1,0].text(i, val + 10000, f'${val/1e6:.1f}M',
                   ha='center', fontsize=9, fontweight='bold')

# Chart 4 — WoW conversion change
wow_values = weekly['conv_wow_change'].fillna(0)
colors_wow = ['#1D9E75' if x >= 0 else '#E24B4A'
              for x in wow_values]
axes[1,1].bar(weeks, wow_values,
              color=colors_wow, edgecolor='white')
axes[1,1].axhline(y=0, color='black', linewidth=0.8)
axes[1,1].set_title('Week over Week Conversion Change',
                     fontweight='bold')
axes[1,1].set_ylabel('Change %')
axes[1,1].set_xlabel('Week Number')
for i, val in enumerate(wow_values):
    axes[1,1].text(i, val + 0.0001,
                   f'+{val:.3f}%' if val >= 0 else f'{val:.3f}%',
                   ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('/kaggle/working/weekly_trend.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Phase 13 complete.")

In [ ]:
# ═══════════════════════════════════════════════
# PHASE 14 — BRAND GAP ANALYSIS
# High views but low purchase brands
# These are brands Constructor would rerank
# ═══════════════════════════════════════════════

# Use brand_funnel from Phase 7
brand_gap = brand_funnel.copy()

# Calculate view to purchase gap
brand_gap['view_purchase_gap'] = brand_gap['view_sessions'] - brand_gap['purchase_sessions']
brand_gap['gap_pct']           = (brand_gap['view_purchase_gap'] / brand_gap['view_sessions'] * 100).round(2)

# High view low conversion brands
# Views > 5000 but conversion < 2%
problem_brands = brand_gap[
    (brand_gap['view_sessions'] >= 5000) &
    (brand_gap['conversion_rate'] < 2.0)
].sort_values('view_sessions', ascending=False)

# High view high conversion brands
strong_brands = brand_gap[
    (brand_gap['view_sessions'] >= 5000) &
    (brand_gap['conversion_rate'] >= 3.0)
].sort_values('conversion_rate', ascending=False)

print("=== PROBLEM BRANDS — High Views Low Conversion ===")
print("These brands appear in search but users don't buy\n")
for idx, row in problem_brands.head(10).iterrows():
    print(f"  {idx.upper()}")
    print(f"    Views      : {int(row['view_sessions']):>8,}")
    print(f"    Purchases  : {int(row['purchase_sessions']):>8,}")
    print(f"    Conversion : {row['conversion_rate']:>7.2f}%")
    print(f"    Gap        : {int(row['view_purchase_gap']):>8,} lost visitors")
    print()

print("=== STRONG BRANDS — High Views High Conversion ===")
print("These brands should be ranked higher in search\n")
for idx, row in strong_brands.head(10).iterrows():
    print(f"  {idx.upper()}")
    print(f"    Views      : {int(row['view_sessions']):>8,}")
    print(f"    Purchases  : {int(row['purchase_sessions']):>8,}")
    print(f"    Conversion : {row['conversion_rate']:>7.2f}%")
    print()

# ── CHART — Brand gap scatter plot ──────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Chart 1 — Problem brands
problem_top10 = problem_brands.head(10)
axes[0].barh(problem_top10.index,
             problem_top10['view_sessions'],
             color='#E8E8E8', edgecolor='white',
             label='Views')
axes[0].barh(problem_top10.index,
             problem_top10['purchase_sessions'],
             color='#E24B4A', edgecolor='white',
             label='Purchases')
axes[0].set_title('Problem Brands\nHigh Views — Low Purchases',
                   fontweight='bold')
axes[0].set_xlabel('Sessions')
axes[0].invert_yaxis()
axes[0].legend()
for i, (idx, row) in enumerate(problem_top10.iterrows()):
    axes[0].text(row['view_sessions'] + 100, i,
                 f"{row['conversion_rate']:.2f}%",
                 va='center', fontsize=9, color='#E24B4A',
                 fontweight='bold')

# Chart 2 — Strong brands
strong_top10 = strong_brands.head(10)
axes[1].barh(strong_top10.index,
             strong_top10['view_sessions'],
             color='#E8E8E8', edgecolor='white',
             label='Views')
axes[1].barh(strong_top10.index,
             strong_top10['purchase_sessions'],
             color='#1D9E75', edgecolor='white',
             label='Purchases')
axes[1].set_title('Strong Brands\nHigh Views — High Purchases',
                   fontweight='bold')
axes[1].set_xlabel('Sessions')
axes[1].invert_yaxis()
axes[1].legend()
for i, (idx, row) in enumerate(strong_top10.iterrows()):
    axes[1].text(row['view_sessions'] + 100, i,
                 f"{row['conversion_rate']:.2f}%",
                 va='center', fontsize=9, color='#1D9E75',
                 fontweight='bold')

plt.suptitle('REES46 October 2019 — Brand Gap Analysis\n(Constructor would rerank problem brands lower)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/brand_gap_analysis.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Phase 14 complete.")

In [ ]:
# ═══════════════════════════════════════════════
# PHASE 15 — EXPORT OCTOBER TABLES
# ═══════════════════════════════════════════════
import os
OUTPUT = '/kaggle/working/'

# Add month column to df_oct
df_oct['month'] = 'October'

# 1. fact_events
df_oct.to_csv(f'{OUTPUT}oct_fact_events.csv', index=False)
print(f"✅ oct_fact_events.csv → {df_oct.shape}")

# 2. session_summary
oct_session_summary = df_oct.groupby('user_session').agg(
    user_id      = ('user_id',    'first'),
    date         = ('date',       'first'),
    week_number  = ('week_number','first'),
    day_of_week  = ('day_of_week','first'),
    viewed       = ('event_type', lambda x: int('view'     in x.values)),
    carted       = ('event_type', lambda x: int('cart'     in x.values)),
    purchased    = ('event_type', lambda x: int('purchase' in x.values)),
    revenue      = ('price',      lambda x: x[df_oct.loc[x.index,'event_type']=='purchase'].sum()),
    total_events = ('event_type', 'count'),
).reset_index()
oct_session_summary['month'] = 'October'
oct_session_summary.to_csv(f'{OUTPUT}oct_session_summary.csv', index=False)
print(f"✅ oct_session_summary.csv → {oct_session_summary.shape}")

# 3. dim_product
oct_dim_product = df_oct.groupby('product_id').agg(
    brand           = ('brand',       'first'),
    category_l1     = ('category_l1', 'first'),
    category_l2     = ('category_l2', 'first'),
    avg_price       = ('price',       'mean'),
    price_tier      = ('price_tier',  'first'),
    total_views     = ('event_type',  lambda x: (x=='view').sum()),
    total_carts     = ('event_type',  lambda x: (x=='cart').sum()),
    total_purchases = ('event_type',  lambda x: (x=='purchase').sum()),
).reset_index()
oct_dim_product['conversion_rate'] = (
    oct_dim_product['total_purchases'] /
    oct_dim_product['total_views'].replace(0,1) * 100
).round(2)
oct_dim_product['month'] = 'October'
oct_dim_product.to_csv(f'{OUTPUT}oct_dim_product.csv', index=False)
print(f"✅ oct_dim_product.csv → {oct_dim_product.shape}")

# 4. dim_user
oct_dim_user = df_oct.groupby('user_id').agg(
    total_sessions  = ('user_session',  'nunique'),
    total_events    = ('event_type',    'count'),
    total_purchases = ('event_type',    lambda x: (x=='purchase').sum()),
    total_spend     = ('price',         lambda x: x[df_oct.loc[x.index,'event_type']=='purchase'].sum()),
    first_seen      = ('date',          'min'),
    last_seen       = ('date',          'max'),
).reset_index()
oct_dim_user['segment'] = pd.cut(
    oct_dim_user['total_sessions'],
    bins   = [0, 1, 5, 99999],
    labels = ['new', 'returning', 'loyal']
)
oct_dim_user['month'] = 'October'
oct_dim_user.to_csv(f'{OUTPUT}oct_dim_user.csv', index=False)
print(f"✅ oct_dim_user.csv → {oct_dim_user.shape}")

# 5. dim_category
oct_dim_category = reliable.copy()
oct_dim_category['revenue'] = oct_dim_category['revenue'].round(2)
oct_dim_category.drop(columns=['revenue_formatted'], errors='ignore', inplace=True)
oct_dim_category['month'] = 'October'
oct_dim_category.to_csv(f'{OUTPUT}oct_dim_category.csv', index=True)
print(f"✅ oct_dim_category.csv → {oct_dim_category.shape}")

# 6. dim_brand
oct_dim_brand = brand_funnel.copy()
oct_dim_brand.drop(columns=['revenue_formatted'], errors='ignore', inplace=True)
oct_dim_brand['month'] = 'October'
oct_dim_brand.to_csv(f'{OUTPUT}oct_dim_brand.csv', index=True)
print(f"✅ oct_dim_brand.csv → {oct_dim_brand.shape}")

# Summary
print("\n" + "="*45)
print("  PHASE 15 COMPLETE — OCTOBER FILES SAVED")
print("="*45)
files = [f for f in os.listdir(OUTPUT) if f.startswith('oct_')]
for f in sorted(files):
    size = os.path.getsize(f'{OUTPUT}{f}') / 1e6
    print(f"  {f:<35} {size:.1f} MB")
print("="*45)
print("\n✅ October Phase 15 complete.")

In [ ]:
dfs = [(name, obj) for name, obj in globals().items() 
       if isinstance(obj, pd.DataFrame)]

if dfs:
    for name, obj in dfs:
        size = obj.memory_usage(deep=True).sum() / 1e9
        print(f"{name} → {obj.shape} → {size:.2f} GB")
else:
    print("No DataFrames in RAM")

In [ ]:
import gc

# Keep only df_oct — delete everything else
keep = ['df_oct']

to_delete = []
for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame) and name not in keep:
        to_delete.append(name)

for name in to_delete:
    del globals()[name]

gc.collect()

# Verify
dfs = [(name, obj) for name, obj in globals().items()
       if isinstance(obj, pd.DataFrame)]
for name, obj in dfs:
    size = obj.memory_usage(deep=True).sum() / 1e9
    print(f"{name} → {obj.shape} → {size:.2f} GB")

# ***NOV Dataset***   

In [ ]:
import pandas as pd
import os

path = '/kaggle/input/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store'

# Check both file sizes
for f in os.listdir(path):
    size = os.path.getsize(f'{path}/{f}') / 1e9
    print(f"{f}  →  {size:.2f} GB")

# Check Kaggle RAM available
import psutil
ram = psutil.virtual_memory()
print(f"\nTotal RAM  : {ram.total/1e9:.1f} GB")
print(f"Available  : {ram.available/1e9:.1f} GB")

In [ ]:
import os
import pandas as pd

# List all files in the downloaded path
files = os.listdir(path)
print(f"Files in dataset: {files}")

# Load only the '2019-Nov.csv' file
csv_file_name = '2019-Nov.csv'
if csv_file_name in files:
    print(f"Loading '{csv_file_name}'")
    df = pd.read_csv(os.path.join(path, csv_file_name), nrows=5)
    display(df)
else:
    print(f"'{csv_file_name}' not found in the directory.")

In [ ]:
import pandas as pd

# Load full November dataset
df_full = pd.read_csv(os.path.join(path, '2019-Nov.csv'))
print(f"Full data shape : {df_full.shape}")
print(f"\nOriginal distribution:")
print((df_full['event_type'].value_counts() / len(df_full) * 100).round(2))
print(f"\nOriginal missing values:")
print(df_full.isnull().sum())

# Stratified sample of 5M rows mirroring original
df_nov = df_full.groupby('event_type', group_keys=False).apply(
    lambda x: x.sample(frac=5_000_000/len(df_full), random_state=42)
).reset_index(drop=True)

del df_full

print(f"\n--- SAMPLE (df1) ---")
print(f"Shape  : {df_nov.shape}")
print(f"Memory : {df_nov.memory_usage(deep=True).sum()/1e9:.2f} GB")
print(f"\nSample distribution:")
print((df_nov['event_type'].value_counts() / len(df_nov) * 100).round(2))
print(f"\nSample missing values:")
print(df_nov.isnull().sum())

In [ ]:
import gc 
del df
gc.collect()

In [ ]:
dfs = [(name, obj) for name, obj in globals().items() 
       if isinstance(obj, pd.DataFrame)]

if dfs:
    for name, obj in dfs:
        size = obj.memory_usage(deep=True).sum() / 1e9
        print(f"{name} → {obj.shape} → {size:.2f} GB")
else:
    print("No DataFrames in RAM")

In [ ]:
print('     =================First 5 rows of Ocotber data set==================\n')
display(df_nov.head())

In [ ]:
print('     =================Last 5 rows of Ocotber data set==================\n')
display(df_nov.head())

In [ ]:
display(df_nov.shape)

In [ ]:
columns = df_nov.columns
print('=== Columns in Nov dataset===')
for col in columns:
    print(f'-{col}')

In [ ]:
# Print Data Type + basic Information
print("===== Data Types =====")
display(df_nov.dtypes)

print("\n=====Basic Info=====")
print(f"Data Range : {df_nov['event_time'].min()} --->  {df_nov['event_time'].max()}")
print(f"Unique users : {df_nov['user_id'].nunique():,}")
print(f"Unique Session : {df_nov['user_session'].nunique():,}")
print(f"Unique Products : {df_nov['product_id'].nunique():,}")
print(f"Unique Brands: {df_nov['brand'].nunique():,}")
print(f"Unique Category:{df_nov['category_code'].nunique():,}")

In [ ]:
display(df_nov['event_type'].value_counts())

In [ ]:
event_count = df_nov['event_type'].value_counts()
total_event = len(df_nov)
event_percentage = (event_count / total_event * 100).round(2)

print(f"Total Number of events : {total_event:,}")
print("\nPercentage of each event type:\n",event_percentage)

In [ ]:
# view to cart rate
view_to_cart_rate = (event_count['cart']/event_count['view']) * 100
print(f'View to Cart Rate is:{view_to_cart_rate:.2f}%')

In [ ]:
# cart to purchase rate
cart_to_purchase_rate = (event_count['purchase']/event_count['cart']) * 100
print(f'Cart to View Rate is:{cart_to_purchase_rate:.2f}%')

In [ ]:
# Missing Value Analysis 
import matplotlib.pyplot as plt
missing_count = df_nov.isnull().sum()
missing_pct = (missing_count / len(df_nov) * 100).round(2)

missing_df = pd.DataFrame({
    'missing_count':missing_count,
    'missing_percent': missing_pct
}).sort_values('missing_count',ascending = False)

print('====== Misssing Value======')
print(missing_df)

# Plot only columns having missing values
missing_only = missing_df[missing_df['missing_percent']>0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# Bar chart
axes[0].bar(missing_only.index, missing_only['missing_percent'],
            color=['#E24B4A', '#F5A623'], edgecolor='white')
axes[0].set_title('Missing Values by Column (%)', fontweight='bold')
axes[0].set_ylabel('Missing %')
axes[0].set_xlabel('Column')
for i, (idx, row) in enumerate(missing_only.iterrows()):
    axes[0].text(i, row['missing_percent'] + 0.3,
                 f"{row['missing_percent']}%",
                 ha='center', fontsize=10)

# Pie chart
axes[1].pie(missing_only['missing_count'],
            labels=missing_only.index,
            autopct='%1.1f%%',
            colors=['#E24B4A', '#F5A623', '#AAAAAA'],
            startangle=90)
axes[1].set_title('Share of Missing Values', fontweight='bold')

plt.suptitle('REES46 November 2019 — Missing Value Analysis',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/kaggle/working/missing_values.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Missing values chart saved.")

In [ ]:
# ═══════════════════════════════════════════════
# STATISTICAL ANALYSIS
# ═══════════════════════════════════════════════
import matplotlib.pyplot as plt 
# 1. Basic statistics
print("=== PRICE STATISTICS ===")
print(df_nov['price'].describe().round(2))

print("\n=== EVENT TYPE DISTRIBUTION ===")
event_count = df_nov['event_type'].value_counts()
event_pct   = (event_count / len(df_nov) * 100).round(2)
event_df    = pd.DataFrame({'count': event_count, 'percent': event_pct})
print(event_df)

print("\n=== USER STATISTICS ===")
sessions_per_user = df_nov.groupby('user_id')['user_session'].nunique()
print(f"Avg sessions per user : {sessions_per_user.mean():.2f}")
print(f"Max sessions per user : {sessions_per_user.max()}")
print(f"Users with 1 session  : {(sessions_per_user==1).sum():,}")
print(f"Users with 2+ sessions: {(sessions_per_user>1).sum():,}")

print("\n=== PRODUCT STATISTICS ===")
views_per_product = df_nov[df_nov['event_type']=='view'].groupby('product_id').size()
print(f"Avg views per product : {views_per_product.mean():.2f}")
print(f"Max views per product : {views_per_product.max():,}")
print(f"Products viewed once  : {(views_per_product==1).sum():,}")

# ── CHARTS ──────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('REES46 October 2019 — Statistical Analysis',
             fontsize=14, fontweight='bold', y=1.02)

# Chart 1 — Event type bar chart
colors = ['#378ADD', '#EF9F27', '#1D9E75']
axes[0,0].bar(event_df.index, event_df['count'], color=colors, edgecolor='white')
axes[0,0].set_title('Event Count by Type', fontweight='bold')
axes[0,0].set_ylabel('Count')
for i, (idx, row) in enumerate(event_df.iterrows()):
    axes[0,0].text(i, row['count'] + 200000,
                   f"{row['percent']}%", ha='center', fontsize=10)

# Chart 2 — Event type pie
axes[0,1].pie(event_df['count'], labels=event_df.index,
              autopct='%1.1f%%', colors=colors, startangle=90)
axes[0,1].set_title('Event Type Share', fontweight='bold')

# Chart 3 — Price distribution
axes[0,2].hist(df_nov[df_nov['price'] < 1000]['price'], bins=50,
               color='#378ADD', edgecolor='white')
axes[0,2].set_title('Price Distribution (< $1000)', fontweight='bold')
axes[0,2].set_xlabel('Price ($)')
axes[0,2].set_ylabel('Count')

# Chart 4 — Price by event type boxplot
df_nov[df_nov['price'] < 500].boxplot(column='price', by='event_type',
                                  ax=axes[1,0], 
                                  boxprops=dict(color='#378ADD'),
                                  medianprops=dict(color='#E24B4A'))
axes[1,0].set_title('Price Distribution by Event Type', fontweight='bold')
axes[1,0].set_xlabel('Event Type')
axes[1,0].set_ylabel('Price ($)')
plt.sca(axes[1,0])
plt.title('Price by Event Type')

# Chart 5 — Sessions per user distribution
sessions_per_user_clipped = sessions_per_user.clip(upper=10)
axes[1,1].hist(sessions_per_user_clipped, bins=10,
               color='#1D9E75', edgecolor='white')
axes[1,1].set_title('Sessions per User (capped at 10)', fontweight='bold')
axes[1,1].set_xlabel('Number of Sessions')
axes[1,1].set_ylabel('Number of Users')

# Chart 6 — Top 10 categories by event count
top_cats = df_nov[df_nov['category_code'] != 'unknown'] \
               .groupby('category_code').size() \
               .sort_values(ascending=False).head(10)
axes[1,2].barh(top_cats.index, top_cats.values,
               color='#7F77DD', edgecolor='white')
axes[1,2].set_title('Top 10 Categories by Events', fontweight='bold')
axes[1,2].set_xlabel('Event Count')
axes[1,2].invert_yaxis()

plt.tight_layout()
plt.savefig('/kaggle/working/statistical_analysis.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Statistical analysis charts saved.")

In [ ]:
# ═══════════════════════════════════════════════
# PHASE 3 — DATA CLEANING
# ═══════════════════════════════════════════════
import gc

print(f"Rows before cleaning : {len(df_nov):,}")

# 1. Convert event_time to datetime
df_nov['event_time'] = pd.to_datetime(df_nov['event_time'], utc=True)

# 2. Remove price = 0
df_nov = df_nov[df_nov['price'] > 0].copy()

# 3. Fill nulls
df_nov['category_code'] = df_nov['category_code'].fillna('unknown')
df_nov['brand']         = df_nov['brand'].fillna('unknown')

# 4. New time columns
df_nov['date']        = df_nov['event_time'].dt.date
df_nov['hour']        = df_nov['event_time'].dt.hour
df_nov['day_of_week'] = df_nov['event_time'].dt.day_name()
df_nov['week_number'] = df_nov['event_time'].dt.isocalendar().week.astype(int)
df_nov['month_name']  = 'November'

# 5. Category levels
df_nov['category_l1'] = df_nov['category_code'].str.split('.').str[0]
df_nov['category_l2'] = df_nov['category_code'].str.split('.').str[1].fillna('unknown')

# 6. Price tier
df_nov['price_tier'] = pd.cut(
    df_nov['price'],
    bins   = [0, 50, 200, 500, 99999],
    labels = ['budget', 'mid', 'premium', 'luxury']
)

# 7. Dtype conversion — save memory
df_nov['product_id']    = df_nov['product_id'].astype('int32')
df_nov['category_id']   = df_nov['category_id'].astype('int32')
df_nov['user_id']       = df_nov['user_id'].astype('int32')
df_nov['price']         = df_nov['price'].astype('float32')
df_nov['hour']          = df_nov['hour'].astype('int8')
df_nov['week_number']   = df_nov['week_number'].astype('int8')
df_nov['event_type']    = df_nov['event_type'].astype('category')
df_nov['brand']         = df_nov['brand'].astype('category')
df_nov['category_code'] = df_nov['category_code'].astype('category')
df_nov['category_l1']   = df_nov['category_l1'].astype('category')
df_nov['category_l2']   = df_nov['category_l2'].astype('category')
df_nov['day_of_week']   = df_nov['day_of_week'].astype('category')
df_nov['price_tier']    = df_nov['price_tier'].astype('category')
df_nov['month_name']    = df_nov['month_name'].astype('category')

gc.collect()

print(f"Rows after cleaning  : {len(df_nov):,}")
print(f"Rows removed         : {5000000 - len(df_nov):,}")
print(f"Memory               : {df_nov.memory_usage(deep=True).sum()/1e9:.2f} GB")
print(f"Total columns        : {df_nov.shape[1]}")
print(f"\nNull check:")
print(df_nov.isnull().sum())
print(f"\nDtypes:")
print(df_nov.dtypes)
print("\n✅ Phase 3 complete. df_nov is clean and ready for analysis.")

In [ ]:
dfs = [(name, obj) for name, obj in globals().items() 
       if isinstance(obj, pd.DataFrame)]

if dfs:
    for name, obj in dfs:
        size = obj.memory_usage(deep=True).sum() / 1e9
        print(f"{name} → {obj.shape} → {size:.2f} GB")
else:
    print("No DataFrames in RAM")

In [ ]:
# ═══════════════════════════════════════════════
# PHASE 4 — FUNNEL ANALYSIS
# ═══════════════════════════════════════════════

# Step 1 — Count unique sessions per funnel stage
total_sessions    = df_nov['user_session'].nunique()
view_sessions     = df_nov[df_nov['event_type']=='view']['user_session'].nunique()
cart_sessions     = df_nov[df_nov['event_type']=='cart']['user_session'].nunique()
purchase_sessions = df_nov[df_nov['event_type']=='purchase']['user_session'].nunique()

# Step 2 — Drop-off at each step
drop_no_view     = total_sessions   - view_sessions
drop_no_cart     = view_sessions    - cart_sessions
drop_no_purchase = cart_sessions    - purchase_sessions

# Step 3 — Rates
view_rate        = view_sessions     / total_sessions * 100
cart_rate        = cart_sessions     / total_sessions * 100
purchase_rate    = purchase_sessions / total_sessions * 100
view_to_cart     = cart_sessions     / view_sessions  * 100
cart_to_purchase = purchase_sessions / cart_sessions  * 100
cart_abandonment = 100 - cart_to_purchase
overall_conv     = purchase_sessions / total_sessions * 100

print("=== FUNNEL COUNTS ===")
print(f"Total Sessions         : {total_sessions:,}")
print(f"Sessions with View     : {view_sessions:,}  ({view_rate:.1f}%)")
print(f"Sessions with Cart     : {cart_sessions:,}   ({cart_rate:.1f}%)")
print(f"Sessions with Purchase : {purchase_sessions:,}   ({purchase_rate:.1f}%)")

print(f"\n=== FUNNEL RATES ===")
print(f"View Rate              : {view_rate:.1f}%")
print(f"Cart Rate              : {cart_rate:.1f}%")
print(f"Purchase Rate          : {purchase_rate:.1f}%")
print(f"View → Cart            : {view_to_cart:.1f}%")
print(f"Cart → Purchase        : {cart_to_purchase:.1f}%")
print(f"Cart Abandonment       : {cart_abandonment:.1f}%")
print(f"Overall Conversion     : {overall_conv:.2f}%")

print(f"\n=== DROP-OFF COUNTS ===")
print(f"Left without viewing   : {drop_no_view:,}")
print(f"Viewed not carted      : {drop_no_cart:,}")
print(f"Carted not purchased   : {drop_no_purchase:,}")

# Step 4 — Funnel chart
fig, ax = plt.subplots(figsize=(10, 6))

stages = ['Sessions', 'Views', 'Carts', 'Purchases']
counts = [total_sessions, view_sessions, cart_sessions, purchase_sessions]
colors = ['#185FA5', '#378ADD', '#1D9E75', '#0F6E56']

bars = ax.barh(stages, counts, color=colors, edgecolor='white', height=0.5)

# Annotate count + percentage
for i, (count, stage) in enumerate(zip(counts, stages)):
    pct = count / total_sessions * 100
    ax.text(count + 5000, i, f'{count:,}  ({pct:.1f}%)',
            va='center', fontsize=11, fontweight='bold')

# Annotate drop-off between steps
drops = [drop_no_view, drop_no_cart, drop_no_purchase]
drop_labels = [f'-{d:,}' for d in drops]
for i, (drop, label) in enumerate(zip(drops, drop_labels)):
    ax.annotate(f'drop: {label}',
                xy=(counts[i+1], i+0.5),
                xytext=(counts[i+1] + counts[0]*0.1, i+0.5),
                fontsize=9, color='#E24B4A',
                arrowprops=dict(arrowstyle='->', color='#E24B4A'))

ax.set_xlabel('Number of Sessions')
ax.set_title('REES46 November 2019 — Conversion Funnel\nSession → View → Cart → Purchase',
             fontweight='bold', fontsize=13)
ax.invert_yaxis()
ax.set_xlim(0, total_sessions * 1.4)

plt.tight_layout()
plt.savefig('/kaggle/working/funnel_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Funnel chart saved.")

In [ ]:
# ═══════════════════════════════════════════════
# PHASE 5 — KPI SUMMARY
# ═══════════════════════════════════════════════

# Revenue calculations
total_revenue    = df_nov[df_nov['event_type']=='purchase']['price'].sum()
total_orders     = purchase_sessions
avg_order_value  = total_revenue / total_orders
revenue_per_visitor = total_revenue / total_sessions

print("=" * 45)
print("   November 2019 — KPI SUMMARY")
print("=" * 45)
print(f"  Total Sessions        : {total_sessions:>12,}")
print(f"  Total Orders          : {total_orders:>12,}")
print(f"  Total Revenue         : ${total_revenue:>12,.2f}")
print("-" * 45)
print(f"  Conversion Rate       : {overall_conv:>11.2f}%")
print(f"  Cart Abandonment Rate : {cart_abandonment:>11.2f}%")
print(f"  View → Cart Rate      : {view_to_cart:>11.2f}%")
print(f"  Cart → Purchase Rate  : {cart_to_purchase:>11.2f}%")
print("-" * 45)
print(f"  Revenue per Visitor   : ${revenue_per_visitor:>12,.2f}")
print(f"  Avg Order Value       : ${avg_order_value:>12,.2f}")
print("=" * 45)

# KPI visual — horizontal bar scorecard
fig, ax = plt.subplots(figsize=(10, 6))

kpis   = ['Conversion Rate', 'View→Cart Rate',
          'Cart→Purchase Rate', 'Cart Abandonment']
values = [overall_conv, view_to_cart,
          cart_to_purchase, cart_abandonment]
colors = ['#1D9E75', '#378ADD', '#1D9E75', '#E24B4A']

bars = ax.barh(kpis, values, color=colors,
               edgecolor='white', height=0.4)

for i, val in enumerate(values):
    ax.text(val + 0.3, i, f'{val:.2f}%',
            va='center', fontsize=12, fontweight='bold')

ax.set_xlabel('Percentage (%)')
ax.set_title('REES46 November  2019 — KPI Dashboard',
             fontweight='bold', fontsize=13)
ax.set_xlim(0, max(values) * 1.3)
ax.invert_yaxis()

plt.tight_layout()
plt.savefig('/kaggle/working/kpi_summary.png', dpi=150, bbox_inches='tight')
plt.show()

# Revenue KPI card
print(f"\n  💰 Total Revenue      : ${total_revenue:,.2f}")
print(f"  🧾 Avg Order Value    : ${avg_order_value:,.2f}")
print(f"  📈 Revenue/Visitor    : ${revenue_per_visitor:,.2f}")
print("\n✅ Phase 5 complete.")

In [ ]:
# ═══════════════════════════════════════════════
# PHASE 6 — CATEGORY ANALYSIS — NOVEMBER
# ═══════════════════════════════════════════════

# Count unique sessions per category per event type
cat_view = df_nov[df_nov['event_type']=='view'].groupby(
    'category_l1', observed=True)['user_session'].nunique().rename('view_sessions')

cat_cart = df_nov[df_nov['event_type']=='cart'].groupby(
    'category_l1', observed=True)['user_session'].nunique().rename('cart_sessions')

cat_purchase = df_nov[df_nov['event_type']=='purchase'].groupby(
    'category_l1', observed=True)['user_session'].nunique().rename('purchase_sessions')

cat_revenue = df_nov[df_nov['event_type']=='purchase'].groupby(
    'category_l1', observed=True)['price'].sum().rename('revenue')

# Combine
nov_category_funnel = pd.concat(
    [cat_view, cat_cart, cat_purchase, cat_revenue], axis=1
).fillna(0)

# Calculate rates
nov_category_funnel['view_to_cart_pct']     = (nov_category_funnel['cart_sessions']     / nov_category_funnel['view_sessions'].replace(0,1)  * 100).round(2)
nov_category_funnel['cart_to_purchase_pct'] = (nov_category_funnel['purchase_sessions'] / nov_category_funnel['cart_sessions'].replace(0,1)  * 100).round(2)
nov_category_funnel['cart_abandonment_pct'] = (100 - nov_category_funnel['cart_to_purchase_pct']).round(2)
nov_category_funnel['conversion_rate']      = (nov_category_funnel['purchase_sessions'] / nov_category_funnel['view_sessions'].replace(0,1)  * 100).round(2)

nov_category_funnel = nov_category_funnel.sort_values('view_sessions', ascending=False)

# Filter reliable only
nov_reliable = nov_category_funnel[
    (nov_category_funnel['view_sessions'] >= 1000) &
    (nov_category_funnel['cart_sessions'] >= nov_category_funnel['purchase_sessions'])
].copy()

nov_reliable['revenue_formatted'] = nov_reliable['revenue'].apply(lambda x: f'${x:,.2f}')

print("=== CATEGORY ANALYSIS — NOVEMBER 2019 ===\n")
for idx, row in nov_reliable.iterrows():
    print(f"  {idx.upper()}")
    print(f"    Views        : {int(row['view_sessions']):>10,}")
    print(f"    Carts        : {int(row['cart_sessions']):>10,}")
    print(f"    Purchases    : {int(row['purchase_sessions']):>10,}")
    print(f"    View→Cart    : {row['view_to_cart_pct']:>9.2f}%")
    print(f"    Cart Abandon : {row['cart_abandonment_pct']:>9.2f}%")
    print(f"    Conversion   : {row['conversion_rate']:>9.2f}%")
    print(f"    Revenue      : {row['revenue_formatted']:>15}")
    print()

# ── CHARTS ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

x     = range(len(nov_reliable))
width = 0.25

axes[0].bar([i-width for i in x], nov_reliable['view_sessions'],
            width=width, label='View', color='#378ADD')
axes[0].bar([i for i in x], nov_reliable['cart_sessions'],
            width=width, label='Cart', color='#EF9F27')
axes[0].bar([i+width for i in x], nov_reliable['purchase_sessions'],
            width=width, label='Purchase', color='#1D9E75')
axes[0].set_xticks(x)
axes[0].set_xticklabels(nov_reliable.index, rotation=45, ha='right', fontsize=9)
axes[0].set_title('Category Funnel — View vs Cart vs Purchase',
                   fontweight='bold')
axes[0].set_ylabel('Sessions')
axes[0].legend()

nov_reliable_sorted = nov_reliable.sort_values('cart_abandonment_pct', ascending=False)
colors = ['#E24B4A' if x > 20 else '#1D9E75'
          for x in nov_reliable_sorted['cart_abandonment_pct']]

axes[1].barh(nov_reliable_sorted.index,
             nov_reliable_sorted['cart_abandonment_pct'],
             color=colors, edgecolor='white')
axes[1].set_title('Cart Abandonment Rate by Category (%)',
                   fontweight='bold')
axes[1].set_xlabel('Abandonment Rate %')
axes[1].invert_yaxis()

for i, val in enumerate(nov_reliable_sorted['cart_abandonment_pct']):
    axes[1].text(val + 0.3, i, f'{val:.1f}%',
                 va='center', fontsize=10)

plt.suptitle('REES46 November 2019 — Category Analysis',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/nov_category_analysis.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Phase 6 November complete.")

In [ ]:
# ═══════════════════════════════════════════════
# PHASE 7 — BRAND ANALYSIS
# ═══════════════════════════════════════════════
import matplotlib.pyplot as plt
# Build brand funnel using unique sessions
brand_view = df_nov[df_nov['event_type']=='view'].groupby(
    'brand', observed=True)['user_session'].nunique().rename('view_sessions')

brand_cart = df_nov[df_nov['event_type']=='cart'].groupby(
    'brand', observed=True)['user_session'].nunique().rename('cart_sessions')

brand_purchase = df_nov[df_nov['event_type']=='purchase'].groupby(
    'brand', observed=True)['user_session'].nunique().rename('purchase_sessions')

brand_revenue = df_nov[df_nov['event_type']=='purchase'].groupby(
    'brand', observed=True)['price'].sum().rename('revenue')

# Combine
brand_funnel = pd.concat(
    [brand_view, brand_cart, brand_purchase, brand_revenue], axis=1
).fillna(0)

# Filter reliable brands — min 500 view sessions
# AND cart >= purchase to avoid negative abandonment
brand_funnel = brand_funnel[
    (brand_funnel['view_sessions'] >= 500) &
    (brand_funnel['cart_sessions'] >= brand_funnel['purchase_sessions'])
].copy()

# Calculate rates
brand_funnel['view_to_cart_pct']     = (brand_funnel['cart_sessions']    / brand_funnel['view_sessions']               * 100).round(2)
brand_funnel['cart_to_purchase_pct'] = (brand_funnel['purchase_sessions']/ brand_funnel['cart_sessions'].replace(0,1)  * 100).round(2)
brand_funnel['cart_abandonment_pct'] = (100 - brand_funnel['cart_to_purchase_pct']).round(2)
brand_funnel['conversion_rate']      = (brand_funnel['purchase_sessions']/ brand_funnel['view_sessions']               * 100).round(2)
brand_funnel['revenue_formatted']    = brand_funnel['revenue'].apply(lambda x: f'${x:,.2f}')

brand_funnel = brand_funnel.sort_values('revenue', ascending=False)

print("=== TOP 15 BRANDS BY REVENUE ===\n")
for idx, row in brand_funnel.head(15).iterrows():
    print(f"  {idx.upper()}")
    print(f"    Views      : {int(row['view_sessions']):>8,}")
    print(f"    Purchases  : {int(row['purchase_sessions']):>8,}")
    print(f"    Conversion : {row['conversion_rate']:>7.2f}%")
    print(f"    Revenue    : {row['revenue_formatted']:>15}")
    print()

# ── CHART 1 — Top 10 brands by revenue ──────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

top10_rev = brand_funnel.head(10)

axes[0].barh(top10_rev.index,
             top10_rev['revenue'],
             color='#378ADD', edgecolor='white')
axes[0].set_title('Top 10 Brands by Revenue',
                   fontweight='bold')
axes[0].set_xlabel('Revenue ($)')
axes[0].invert_yaxis()
for i, val in enumerate(top10_rev['revenue']):
    axes[0].text(val + 1000, i, f'${val:,.0f}',
                 va='center', fontsize=9)

# ── CHART 2 — Brand gap: high views low conversion ──
# Brands with many views but low conversion
# These are the brands Constructor would rerank
brand_gap = brand_funnel[
    brand_funnel['view_sessions'] >= 1000
].sort_values('conversion_rate', ascending=True).head(10)

axes[1].barh(brand_gap.index,
             brand_gap['conversion_rate'],
             color='#E24B4A', edgecolor='white')
axes[1].set_title('Brands with Lowest Conversion Rate\n(High Views, Low Purchase)',
                   fontweight='bold')
axes[1].set_xlabel('Conversion Rate %')
axes[1].invert_yaxis()
for i, val in enumerate(brand_gap['conversion_rate']):
    axes[1].text(val + 0.05, i, f'{val:.2f}%',
                 va='center', fontsize=9)

plt.suptitle('REES46 November 2019 — Brand Analysis',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/brand_analysis.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Phase 7 complete.")

In [ ]:
# ═══════════════════════════════════════════════
# PHASE 8 — PRICE TIER ANALYSIS
# ═══════════════════════════════════════════════

# Build price tier funnel using unique sessions
price_view = df_nov[df_nov['event_type']=='view'].groupby(
    'price_tier', observed=True)['user_session'].nunique().rename('view_sessions')

price_cart = df_nov[df_nov['event_type']=='cart'].groupby(
    'price_tier', observed=True)['user_session'].nunique().rename('cart_sessions')

price_purchase = df_nov[df_nov['event_type']=='purchase'].groupby(
    'price_tier', observed=True)['user_session'].nunique().rename('purchase_sessions')

price_revenue = df_nov[df_nov['event_type']=='purchase'].groupby(
    'price_tier', observed=True)['price'].sum().rename('revenue')

# Combine
price_funnel = pd.concat(
    [price_view, price_cart, price_purchase, price_revenue], axis=1
).fillna(0)

# Calculate rates
price_funnel['view_to_cart_pct']     = (price_funnel['cart_sessions']     / price_funnel['view_sessions'].replace(0,1)  * 100).round(2)
price_funnel['cart_to_purchase_pct'] = (price_funnel['purchase_sessions'] / price_funnel['cart_sessions'].replace(0,1)  * 100).round(2)
price_funnel['cart_abandonment_pct'] = (100 - price_funnel['cart_to_purchase_pct']).round(2)
price_funnel['conversion_rate']      = (price_funnel['purchase_sessions'] / price_funnel['view_sessions'].replace(0,1)  * 100).round(2)
price_funnel['revenue_formatted']    = price_funnel['revenue'].apply(lambda x: f'${x:,.2f}')

# Price range labels
price_ranges = {
    'budget'  : '$0-50',
    'mid'     : '$50-200',
    'premium' : '$200-500',
    'luxury'  : '$500+'
}

print("=== PRICE TIER ANALYSIS — NOVEMBER 2019 ===\n")
for idx, row in price_funnel.iterrows():
    tier  = str(idx)
    range_label = price_ranges.get(tier, '')
    print(f"  {tier.upper()} ({range_label})")
    print(f"    Views        : {int(row['view_sessions']):>10,}")
    print(f"    Carts        : {int(row['cart_sessions']):>10,}")
    print(f"    Purchases    : {int(row['purchase_sessions']):>10,}")
    print(f"    View→Cart    : {row['view_to_cart_pct']:>9.2f}%")
    print(f"    Cart Abandon : {row['cart_abandonment_pct']:>9.2f}%")
    print(f"    Conversion   : {row['conversion_rate']:>9.2f}%")
    print(f"    Revenue      : {row['revenue_formatted']:>18}")
    print()

# ── CHART — Price tier comparison ───────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

tiers  = price_funnel.index.astype(str)
colors = ['#1D9E75', '#378ADD', '#EF9F27', '#E24B4A']

# Chart 1 — Conversion rate by tier
axes[0].bar(tiers, price_funnel['conversion_rate'],
            color=colors, edgecolor='white')
axes[0].set_title('Conversion Rate by Price Tier', fontweight='bold')
axes[0].set_ylabel('Conversion Rate %')
axes[0].set_xlabel('Price Tier')
for i, val in enumerate(price_funnel['conversion_rate']):
    axes[0].text(i, val + 0.05, f'{val:.2f}%',
                 ha='center', fontsize=10, fontweight='bold')

# Chart 2 — Cart abandonment by tier
axes[1].bar(tiers, price_funnel['cart_abandonment_pct'],
            color=colors, edgecolor='white')
axes[1].set_title('Cart Abandonment by Price Tier', fontweight='bold')
axes[1].set_ylabel('Abandonment Rate %')
axes[1].set_xlabel('Price Tier')
for i, val in enumerate(price_funnel['cart_abandonment_pct']):
    axes[1].text(i, val + 0.5, f'{val:.1f}%',
                 ha='center', fontsize=10, fontweight='bold')

# Chart 3 — Revenue by tier
axes[2].bar(tiers, price_funnel['revenue'],
            color=colors, edgecolor='white')
axes[2].set_title('Revenue by Price Tier', fontweight='bold')
axes[2].set_ylabel('Revenue ($)')
axes[2].set_xlabel('Price Tier')
for i, val in enumerate(price_funnel['revenue']):
    axes[2].text(i, val + 10000, f'${val/1e6:.1f}M',
                 ha='center', fontsize=10, fontweight='bold')

plt.suptitle('REES46 November 2019 — Price Tier Analysis',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/price_tier_analysis.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Phase 8 complete.")

In [ ]:
# ═══════════════════════════════════════════════
# PHASE 9 — TIME PATTERNS
# ═══════════════════════════════════════════════

# ── Hourly analysis ──────────────────────────────────
hourly = df_nov.groupby(['hour', 'event_type'],
                         observed=True).size().unstack(fill_value=0)

# ── Day of week analysis ─────────────────────────────
day_order = ['Monday','Tuesday','Wednesday',
             'Thursday','Friday','Saturday','Sunday']
dow = df_nov.groupby(['day_of_week', 'event_type'],
                      observed=True).size().unstack(fill_value=0)
dow = dow.reindex(day_order)

print("=== HOURLY PURCHASE PATTERN ===")
print(hourly['purchase'].sort_values(ascending=False).head(5))
print(f"\nPeak purchase hour : {hourly['purchase'].idxmax()}:00")
print(f"Lowest hour        : {hourly['purchase'].idxmin()}:00")

print("\n=== DAY OF WEEK PATTERN ===")
print(dow['purchase'].sort_values(ascending=False))
print(f"\nBest day  : {dow['purchase'].idxmax()}")
print(f"Worst day : {dow['purchase'].idxmin()}")

# ── CHARTS ───────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle('REES46 November 2019 — Time Patterns',
             fontsize=14, fontweight='bold')

# Chart 1 — Purchases by hour
axes[0,0].plot(hourly.index, hourly['purchase'],
               color='#1D9E75', linewidth=2.5, marker='o', markersize=5)
axes[0,0].fill_between(hourly.index, hourly['purchase'],
                        alpha=0.15, color='#1D9E75')
axes[0,0].set_title('Purchases by Hour of Day', fontweight='bold')
axes[0,0].set_xlabel('Hour (UTC)')
axes[0,0].set_ylabel('Purchase Count')
axes[0,0].set_xticks(range(0, 24, 2))
peak_hour = hourly['purchase'].idxmax()
axes[0,0].axvline(x=peak_hour, color='#E24B4A',
                   linestyle='--', alpha=0.7,
                   label=f'Peak: {peak_hour}:00')
axes[0,0].legend()

# Chart 2 — Views, carts, purchases by hour
axes[0,1].plot(hourly.index, hourly['view'],
               color='#378ADD', linewidth=2, label='View')
axes[0,1].plot(hourly.index, hourly['cart'],
               color='#EF9F27', linewidth=2, label='Cart')
axes[0,1].plot(hourly.index, hourly['purchase'],
               color='#1D9E75', linewidth=2, label='Purchase')
axes[0,1].set_title('All Events by Hour of Day', fontweight='bold')
axes[0,1].set_xlabel('Hour (UTC)')
axes[0,1].set_ylabel('Event Count')
axes[0,1].set_xticks(range(0, 24, 2))
axes[0,1].legend()

# Chart 3 — Purchases by day of week
colors_dow = ['#E24B4A' if d in ['Saturday','Sunday']
              else '#378ADD' for d in day_order]
axes[1,0].bar(dow.index, dow['purchase'],
              color=colors_dow, edgecolor='white')
axes[1,0].set_title('Purchases by Day of Week\n(Red = Weekend)',
                     fontweight='bold')
axes[1,0].set_xlabel('Day')
axes[1,0].set_ylabel('Purchase Count')
axes[1,0].tick_params(axis='x', rotation=45)
for i, val in enumerate(dow['purchase']):
    axes[1,0].text(i, val + 50, f'{val:,}',
                   ha='center', fontsize=9, fontweight='bold')

# Chart 4 — All events by day of week
x     = range(len(day_order))
width = 0.25
axes[1,1].bar([i-width for i in x], dow['view'],
              width=width, label='View', color='#378ADD')
axes[1,1].bar([i for i in x], dow['cart'],
              width=width, label='Cart', color='#EF9F27')
axes[1,1].bar([i+width for i in x], dow['purchase'],
              width=width, label='Purchase', color='#1D9E75')
axes[1,1].set_title('All Events by Day of Week', fontweight='bold')
axes[1,1].set_xticks(x)
axes[1,1].set_xticklabels(day_order, rotation=45, ha='right')
axes[1,1].set_ylabel('Count')
axes[1,1].legend()

plt.tight_layout()
plt.savefig('/kaggle/working/time_patterns.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Phase 9 complete.")

In [ ]:
# ═══════════════════════════════════════════════
# PHASE 10 — USER SEGMENTATION
# ═══════════════════════════════════════════════

# Build user level summary
user_summary = df_nov.groupby('user_id', observed=True).agg(
    total_sessions  = ('user_session', 'nunique'),
    total_events    = ('event_type',   'count'),
    total_spend     = ('price',        lambda x: x[df_nov.loc[x.index, 'event_type'] == 'purchase'].sum()),
    total_purchases = ('event_type',   lambda x: (x == 'purchase').sum()),
    total_views     = ('event_type',   lambda x: (x == 'view').sum()),
    total_carts     = ('event_type',   lambda x: (x == 'cart').sum()),
).reset_index()

# Segment users
user_summary['segment'] = pd.cut(
    user_summary['total_sessions'],
    bins   = [0, 1, 5, 99999],
    labels = ['new', 'returning', 'loyal']
)

print("=== USER SEGMENT COUNTS ===")
print(user_summary['segment'].value_counts())

# Segment level KPIs
segment_kpis = user_summary.groupby('segment', observed=True).agg(
    users           = ('user_id',       'count'),
    avg_sessions    = ('total_sessions','mean'),
    avg_spend       = ('total_spend',   'mean'),
    avg_purchases   = ('total_purchases','mean'),
    total_revenue   = ('total_spend',   'sum'),
).round(2).reset_index()

print("\n=== SEGMENT KPIs ===")
for _, row in segment_kpis.iterrows():
    print(f"\n  {str(row['segment']).upper()}")
    print(f"    Users         : {int(row['users']):>10,}")
    print(f"    Avg Sessions  : {row['avg_sessions']:>10.2f}")
    print(f"    Avg Spend     : ${row['avg_spend']:>10.2f}")
    print(f"    Avg Purchases : {row['avg_purchases']:>10.2f}")
    print(f"    Total Revenue : ${row['total_revenue']:>12,.2f}")

# ── CHARTS ───────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('REES46 November 2019 — User Segmentation',
             fontsize=14, fontweight='bold')

colors = ['#378ADD', '#EF9F27', '#E24B4A']
segments = segment_kpis['segment'].astype(str)

# Chart 1 — User count by segment
axes[0].bar(segments, segment_kpis['users'],
            color=colors, edgecolor='white')
axes[0].set_title('Users by Segment', fontweight='bold')
axes[0].set_ylabel('User Count')
axes[0].set_xlabel('Segment')
for i, val in enumerate(segment_kpis['users']):
    axes[0].text(i, val + 1000, f'{int(val):,}',
                 ha='center', fontsize=10, fontweight='bold')

# Chart 2 — Avg spend by segment
axes[1].bar(segments, segment_kpis['avg_spend'],
            color=colors, edgecolor='white')
axes[1].set_title('Avg Spend per User by Segment',
                   fontweight='bold')
axes[1].set_ylabel('Avg Spend ($)')
axes[1].set_xlabel('Segment')
for i, val in enumerate(segment_kpis['avg_spend']):
    axes[1].text(i, val + 0.5, f'${val:.2f}',
                 ha='center', fontsize=10, fontweight='bold')

# Chart 3 — Total revenue by segment
axes[2].bar(segments, segment_kpis['total_revenue'],
            color=colors, edgecolor='white')
axes[2].set_title('Total Revenue by Segment',
                   fontweight='bold')
axes[2].set_ylabel('Revenue ($)')
axes[2].set_xlabel('Segment')
for i, val in enumerate(segment_kpis['total_revenue']):
    axes[2].text(i, val + 1000, f'${val/1e6:.1f}M',
                 ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('/kaggle/working/user_segmentation.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Phase 10 complete.")

In [ ]:
# ═══════════════════════════════════════════════
# PHASE 11 — COHORT RETENTION
# ═══════════════════════════════════════════════
import seaborn as sns 
# First purchase week per user
first_purchase = df_nov[df_nov['event_type']=='purchase'] \
    .groupby('user_id', observed=True)['week_number'] \
    .min().reset_index()
first_purchase.columns = ['user_id', 'cohort_week']

# All purchase events with week
purchases = df_nov[df_nov['event_type']=='purchase'][
    ['user_id','week_number']
].copy()

# Join to get cohort week
purchases = purchases.merge(first_purchase, on='user_id', how='left')
purchases['week_number'] = purchases['week_number'].astype(int)
purchases['cohort_week'] = purchases['cohort_week'].astype(int)
purchases['week_offset'] = purchases['week_number'] - purchases['cohort_week']

# Build cohort matrix
cohort_matrix = purchases.groupby(
    ['cohort_week','week_offset']
)['user_id'].nunique().unstack(fill_value=0)

# Convert to retention % — divide by week 0 cohort size
cohort_size   = cohort_matrix[0]
retention_pct = cohort_matrix.divide(cohort_size, axis=0) * 100

print("=== COHORT RETENTION MATRIX ===")
print(retention_pct.round(1))

# ── CHART — Cohort heatmap ───────────────────────────
fig, ax = plt.subplots(figsize=(14, 7))

sns.heatmap(
    retention_pct.round(1),
    annot    = True,
    fmt      = '.1f',
    cmap     = 'Blues',
    ax       = ax,
    linewidths = 0.5,
    cbar_kws = {'label': 'Retention %'}
)

ax.set_title('REES46 November 2019 — Cohort Retention Matrix\n(% of week 0 cohort returning)',
             fontweight='bold', fontsize=13)
ax.set_xlabel('Weeks Since First Purchase')
ax.set_ylabel('Cohort Week')

plt.tight_layout()
plt.savefig('/kaggle/working/cohort_retention.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Phase 11 complete.")

In [ ]:
# ═══════════════════════════════════════════════
# PHASE 12 — CO-PURCHASE ANALYSIS
# ═══════════════════════════════════════════════

# Get all purchase events with product and session
copurchase = df_nov[df_nov['event_type']=='purchase'][
    ['user_session','product_id','category_l1','brand','price']
].copy()

# Find sessions with multiple purchases
session_purchase_count = copurchase.groupby(
    'user_session')['product_id'].count()
multi_purchase_sessions = session_purchase_count[
    session_purchase_count >= 2
].index

print(f"Sessions with 1 purchase  : {(session_purchase_count==1).sum():,}")
print(f"Sessions with 2+ purchases: {len(multi_purchase_sessions):,}")

# Get products bought together in same session
multi = copurchase[
    copurchase['user_session'].isin(multi_purchase_sessions)
].copy()

# Self join to find pairs
pairs = multi.merge(multi, on='user_session', suffixes=('_1','_2'))

# Remove same product pairs
pairs = pairs[pairs['product_id_1'] < pairs['product_id_2']]

# Count co-purchases
copurchase_counts = pairs.groupby(
    ['category_l1_1','category_l1_2']
).size().reset_index(name='count')

copurchase_counts = copurchase_counts.sort_values(
    'count', ascending=False).head(15)

print("\n=== TOP 15 CO-PURCHASED CATEGORY PAIRS ===\n")
for _, row in copurchase_counts.iterrows():
    print(f"  {row['category_l1_1'].upper()} + {row['category_l1_2'].upper()}")
    print(f"    Co-purchased : {int(row['count']):,} times")
    print()

# ── CHART — Co-purchase bar chart instead of heatmap ─
fig, ax = plt.subplots(figsize=(12, 7))

# Clean unknown pairs
copurchase_clean = copurchase_counts[
    (copurchase_counts['category_l1_1'] != 'unknown') &
    (copurchase_counts['category_l1_2'] != 'unknown')
].head(10).copy()

copurchase_clean['pair'] = copurchase_clean['category_l1_1'].str.upper() + \
                           ' + ' + \
                           copurchase_clean['category_l1_2'].str.upper()

ax.barh(copurchase_clean['pair'],
        copurchase_clean['count'],
        color='#378ADD', edgecolor='white')
ax.set_title('REES46 November 2019 — Top Co-Purchased Category Pairs',
             fontweight='bold', fontsize=13)
ax.set_xlabel('Co-purchase Count')
ax.invert_yaxis()

for i, val in enumerate(copurchase_clean['count']):
    ax.text(val + 0.5, i, f'{int(val):,}',
            va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('/kaggle/working/copurchase_analysis.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Phase 12 complete.")

In [ ]:
# ═══════════════════════════════════════════════
# PHASE 13 — WEEKLY TREND
# ═══════════════════════════════════════════════

# Weekly aggregation
weekly = df_nov.groupby(['week_number', 'event_type'],
                         observed=True).size().unstack(fill_value=0)

# Add revenue
weekly_revenue = df_nov[df_nov['event_type']=='purchase'] \
    .groupby('week_number', observed=True)['price'].sum()

weekly['revenue'] = weekly_revenue
weekly['revenue'] = weekly['revenue'].fillna(0)

# Calculate weekly KPIs
weekly['total_events']    = weekly['view'] + weekly['cart'] + weekly['purchase']
weekly['conversion_rate'] = (weekly['purchase'] / weekly['total_events'] * 100).round(3)
weekly['cart_abandonment']= ((weekly['cart'] - weekly['purchase']) / weekly['cart'].replace(0,1) * 100).round(2)
weekly['revenue_formatted']= weekly['revenue'].apply(lambda x: f'${x:,.2f}')

# Week over week change
weekly['conv_wow_change'] = weekly['conversion_rate'].diff().round(3)
weekly['rev_wow_change']  = weekly['revenue'].diff().round(2)

print("=== WEEKLY TREND — NOVEMBER 2019 ===\n")
for idx, row in weekly.iterrows():
    wow = f"+{row['conv_wow_change']:.3f}%" if row['conv_wow_change'] > 0 \
          else f"{row['conv_wow_change']:.3f}%"
    print(f"  Week {idx}")
    print(f"    Views      : {int(row['view']):>10,}")
    print(f"    Carts      : {int(row['cart']):>10,}")
    print(f"    Purchases  : {int(row['purchase']):>10,}")
    print(f"    Conv Rate  : {row['conversion_rate']:>9.3f}%")
    print(f"    WoW Change : {wow:>10}")
    print(f"    Revenue    : {row['revenue_formatted']:>18}")
    print()

# ── CHARTS ───────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(18, 10))
fig.suptitle('REES46 November 2019 — Weekly Trend',
             fontsize=14, fontweight='bold')

weeks = weekly.index.astype(str)

# Chart 1 — Weekly purchases
axes[0,0].plot(weeks, weekly['purchase'],
               color='#1D9E75', linewidth=2.5,
               marker='o', markersize=8)
axes[0,0].fill_between(weeks, weekly['purchase'],
                        alpha=0.15, color='#1D9E75')
axes[0,0].set_title('Weekly Purchases', fontweight='bold')
axes[0,0].set_ylabel('Purchase Count')
axes[0,0].set_xlabel('Week Number')
for i, val in enumerate(weekly['purchase']):
    axes[0,0].text(i, val + 100, f'{int(val):,}',
                   ha='center', fontsize=9, fontweight='bold')

# Chart 2 — Weekly conversion rate
colors_conv = ['#1D9E75' if x >= 0 else '#E24B4A'
               for x in weekly['conv_wow_change'].fillna(0)]
axes[0,1].bar(weeks, weekly['conversion_rate'],
              color='#378ADD', edgecolor='white')
axes[0,1].set_title('Weekly Conversion Rate', fontweight='bold')
axes[0,1].set_ylabel('Conversion Rate %')
axes[0,1].set_xlabel('Week Number')
for i, val in enumerate(weekly['conversion_rate']):
    axes[0,1].text(i, val + 0.001, f'{val:.3f}%',
                   ha='center', fontsize=9, fontweight='bold')

# Chart 3 — Weekly revenue
axes[1,0].bar(weeks, weekly['revenue'],
              color='#EF9F27', edgecolor='white')
axes[1,0].set_title('Weekly Revenue', fontweight='bold')
axes[1,0].set_ylabel('Revenue ($)')
axes[1,0].set_xlabel('Week Number')
for i, val in enumerate(weekly['revenue']):
    axes[1,0].text(i, val + 10000, f'${val/1e6:.1f}M',
                   ha='center', fontsize=9, fontweight='bold')

# Chart 4 — WoW conversion change
wow_values = weekly['conv_wow_change'].fillna(0)
colors_wow = ['#1D9E75' if x >= 0 else '#E24B4A'
              for x in wow_values]
axes[1,1].bar(weeks, wow_values,
              color=colors_wow, edgecolor='white')
axes[1,1].axhline(y=0, color='black', linewidth=0.8)
axes[1,1].set_title('Week over Week Conversion Change',
                     fontweight='bold')
axes[1,1].set_ylabel('Change %')
axes[1,1].set_xlabel('Week Number')
for i, val in enumerate(wow_values):
    axes[1,1].text(i, val + 0.0001,
                   f'+{val:.3f}%' if val >= 0 else f'{val:.3f}%',
                   ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('/kaggle/working/weekly_trend.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Phase 13 complete.")

In [ ]:
# ═══════════════════════════════════════════════
# PHASE 14 — BRAND GAP ANALYSIS
# High views but low purchase brands
# These are brands Constructor would rerank
# ═══════════════════════════════════════════════

# Use brand_funnel from Phase 7
brand_gap = brand_funnel.copy()

# Calculate view to purchase gap
brand_gap['view_purchase_gap'] = brand_gap['view_sessions'] - brand_gap['purchase_sessions']
brand_gap['gap_pct']           = (brand_gap['view_purchase_gap'] / brand_gap['view_sessions'] * 100).round(2)

# High view low conversion brands
# Views > 5000 but conversion < 2%
problem_brands = brand_gap[
    (brand_gap['view_sessions'] >= 5000) &
    (brand_gap['conversion_rate'] < 2.0)
].sort_values('view_sessions', ascending=False)

# High view high conversion brands
strong_brands = brand_gap[
    (brand_gap['view_sessions'] >= 5000) &
    (brand_gap['conversion_rate'] >= 3.0)
].sort_values('conversion_rate', ascending=False)

print("=== PROBLEM BRANDS — High Views Low Conversion ===")
print("These brands appear in search but users don't buy\n")
for idx, row in problem_brands.head(10).iterrows():
    print(f"  {idx.upper()}")
    print(f"    Views      : {int(row['view_sessions']):>8,}")
    print(f"    Purchases  : {int(row['purchase_sessions']):>8,}")
    print(f"    Conversion : {row['conversion_rate']:>7.2f}%")
    print(f"    Gap        : {int(row['view_purchase_gap']):>8,} lost visitors")
    print()

print("=== STRONG BRANDS — High Views High Conversion ===")
print("These brands should be ranked higher in search\n")
for idx, row in strong_brands.head(10).iterrows():
    print(f"  {idx.upper()}")
    print(f"    Views      : {int(row['view_sessions']):>8,}")
    print(f"    Purchases  : {int(row['purchase_sessions']):>8,}")
    print(f"    Conversion : {row['conversion_rate']:>7.2f}%")
    print()

# ── CHART — Brand gap scatter plot ──────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Chart 1 — Problem brands
problem_top10 = problem_brands.head(10)
axes[0].barh(problem_top10.index,
             problem_top10['view_sessions'],
             color='#E8E8E8', edgecolor='white',
             label='Views')
axes[0].barh(problem_top10.index,
             problem_top10['purchase_sessions'],
             color='#E24B4A', edgecolor='white',
             label='Purchases')
axes[0].set_title('Problem Brands\nHigh Views — Low Purchases',
                   fontweight='bold')
axes[0].set_xlabel('Sessions')
axes[0].invert_yaxis()
axes[0].legend()
for i, (idx, row) in enumerate(problem_top10.iterrows()):
    axes[0].text(row['view_sessions'] + 100, i,
                 f"{row['conversion_rate']:.2f}%",
                 va='center', fontsize=9, color='#E24B4A',
                 fontweight='bold')

# Chart 2 — Strong brands
strong_top10 = strong_brands.head(10)
axes[1].barh(strong_top10.index,
             strong_top10['view_sessions'],
             color='#E8E8E8', edgecolor='white',
             label='Views')
axes[1].barh(strong_top10.index,
             strong_top10['purchase_sessions'],
             color='#1D9E75', edgecolor='white',
             label='Purchases')
axes[1].set_title('Strong Brands\nHigh Views — High Purchases',
                   fontweight='bold')
axes[1].set_xlabel('Sessions')
axes[1].invert_yaxis()
axes[1].legend()
for i, (idx, row) in enumerate(strong_top10.iterrows()):
    axes[1].text(row['view_sessions'] + 100, i,
                 f"{row['conversion_rate']:.2f}%",
                 va='center', fontsize=9, color='#1D9E75',
                 fontweight='bold')

plt.suptitle('REES46 November 2019 — Brand Gap Analysis\n(Constructor would rerank problem brands lower)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/brand_gap_analysis.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Phase 14 complete.")

In [ ]:
for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame):
        size = obj.memory_usage(deep=True).sum() / 1e9
        print(f"{name} → {obj.shape} → {size:.2f} GB")

In [ ]:
import pandas as pd

dfs = [(name, obj) for name, obj in globals().items()
       if isinstance(obj, pd.DataFrame)]

if dfs:
    for name, obj in dfs:
        size = obj.memory_usage(deep=True).sum() / 1e9
        print(f"{name} → {obj.shape} → {size:.2f} GB")
else:
    print("No DataFrames in RAM")

In [ ]:
# ═══════════════════════════════════════════════
# PHASE 15 — EXPORT NOVEMBER TABLES
# ═══════════════════════════════════════════════

import os
OUTPUT = '/kaggle/working/'

# Add month column to df_nov
df_nov['month'] = 'November'

# 1. fact_events
df_nov.to_csv(f'{OUTPUT}nov_fact_events.csv', index=False)
print(f"✅ nov_fact_events.csv → {df_nov.shape}")

# 2. session_summary
nov_session_summary = df_nov.groupby('user_session').agg(
    user_id      = ('user_id',    'first'),
    date         = ('date',       'first'),
    week_number  = ('week_number','first'),
    day_of_week  = ('day_of_week','first'),
    viewed       = ('event_type', lambda x: int('view'     in x.values)),
    carted       = ('event_type', lambda x: int('cart'     in x.values)),
    purchased    = ('event_type', lambda x: int('purchase' in x.values)),
    revenue      = ('price',      lambda x: x[df_nov.loc[x.index,'event_type']=='purchase'].sum()),
    total_events = ('event_type', 'count'),
).reset_index()
nov_session_summary['month'] = 'November'
nov_session_summary.to_csv(f'{OUTPUT}nov_session_summary.csv', index=False)
print(f"✅ nov_session_summary.csv → {nov_session_summary.shape}")

# 3. dim_product
nov_dim_product = df_nov.groupby('product_id').agg(
    brand           = ('brand',       'first'),
    category_l1     = ('category_l1', 'first'),
    category_l2     = ('category_l2', 'first'),
    avg_price       = ('price',       'mean'),
    price_tier      = ('price_tier',  'first'),
    total_views     = ('event_type',  lambda x: (x=='view').sum()),
    total_carts     = ('event_type',  lambda x: (x=='cart').sum()),
    total_purchases = ('event_type',  lambda x: (x=='purchase').sum()),
).reset_index()
nov_dim_product['conversion_rate'] = (
    nov_dim_product['total_purchases'] /
    nov_dim_product['total_views'].replace(0,1) * 100
).round(2)
nov_dim_product['month'] = 'November'
nov_dim_product.to_csv(f'{OUTPUT}nov_dim_product.csv', index=False)
print(f"✅ nov_dim_product.csv → {nov_dim_product.shape}")

# 4. dim_user
nov_dim_user = user_summary.copy()
nov_dim_user['segment'] = pd.cut(
    nov_dim_user['total_sessions'],
    bins   = [0, 1, 5, 99999],
    labels = ['new', 'returning', 'loyal']
)
nov_dim_user['month'] = 'November'
nov_dim_user.to_csv(f'{OUTPUT}nov_dim_user.csv', index=False)
print(f"✅ nov_dim_user.csv → {nov_dim_user.shape}")

# 5. dim_category
nov_dim_category = nov_reliable.copy()
nov_dim_category['revenue'] = nov_dim_category['revenue'].round(2)
nov_dim_category.drop(columns=['revenue_formatted'], errors='ignore', inplace=True)
nov_dim_category['month'] = 'November'
nov_dim_category.to_csv(f'{OUTPUT}nov_dim_category.csv', index=True)
print(f"✅ nov_dim_category.csv → {nov_dim_category.shape}")

# 6. dim_brand
nov_dim_brand = brand_funnel.copy()
nov_dim_brand.drop(columns=['revenue_formatted'], errors='ignore', inplace=True)
nov_dim_brand['month'] = 'November'
nov_dim_brand.to_csv(f'{OUTPUT}nov_dim_brand.csv', index=True)
print(f"✅ nov_dim_brand.csv → {nov_dim_brand.shape}")

# Summary
print("\n" + "="*45)
print("  PHASE 15 COMPLETE — NOVEMBER FILES SAVED")
print("="*45)
files = [f for f in os.listdir(OUTPUT) if f.startswith('nov_')]
for f in sorted(files):
    size = os.path.getsize(f'{OUTPUT}{f}') / 1e6
    print(f"  {f:<35} {size:.1f} MB")
print("="*45)
print("\n✅ November Phase 15 complete.")

# **Combine Analysis**

In [ ]:
import os

# Check both datasets are accessible
oct_path = '/kaggle/input/datasets/piyushxx7/rees46-oct-exports'
nov_path = '/kaggle/input/datasets/piyushxx7/rees46-nov-exports'

print("=== OCTOBER FILES ===")
for f in os.listdir(oct_path):
    size = os.path.getsize(f'{oct_path}/{f}') / 1e6
    print(f"  {f:<40} {size:.1f} MB")

print("\n=== NOVEMBER FILES ===")
for f in os.listdir(nov_path):
    size = os.path.getsize(f'{nov_path}/{f}') / 1e6
    print(f"  {f:<40} {size:.1f} MB")

In [ ]:
import pandas as pd
import os

oct_path = '/kaggle/input/datasets/piyushxx7/rees46-oct-exports'
nov_path = '/kaggle/input/datasets/piyushxx7/rees46-nov-exports'
OUTPUT   = '/kaggle/working/'

tables = ['session_summary', 'dim_product',
          'dim_user', 'dim_brand', 'dim_category']

for table in tables:
    oct_df   = pd.read_csv(f'{oct_path}/oct_{table}.csv')
    nov_df   = pd.read_csv(f'{nov_path}/nov_{table}.csv')
    combined = pd.concat([oct_df, nov_df], ignore_index=True)
    combined.to_csv(f'{OUTPUT}combined_{table}.csv', index=False)
    print(f"✅ combined_{table}.csv → {combined.shape}")

print("\n=== ALL FILES IN OUTPUT ===")
for f in sorted(os.listdir(OUTPUT)):
    if f.endswith('.csv'):
        size = os.path.getsize(f'{OUTPUT}{f}') / 1e6
        print(f"  {f:<40} {size:.1f} MB")

# **SQL**

In [ ]:
!pip install duckdb -q

import os
import pandas as pd
import duckdb

In [ ]:
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
import pandas as pd
import duckdb

base_path = "/kaggle/input/datasets/piyushxx7/ress46-oct-nov-combine-data/"

combined_session_summary = pd.read_csv(base_path + "combined_session_summary.csv")
combined_dim_user = pd.read_csv(base_path + "combined_dim_user.csv")
combined_dim_brand = pd.read_csv(base_path + "combined_dim_brand.csv")
combined_dim_product = pd.read_csv(base_path + "combined_dim_product.csv")
combined_dim_category = pd.read_csv(base_path + "combined_dim_category.csv")

print("Loaded successfully")
print("combined_session_summary:", combined_session_summary.shape)
print("combined_dim_user:", combined_dim_user.shape)
print("combined_dim_brand:", combined_dim_brand.shape)
print("combined_dim_product:", combined_dim_product.shape)
print("combined_dim_category:", combined_dim_category.shape)

In [ ]:
con = duckdb.connect()

con.register("combined_session_summary", combined_session_summary)
con.register("combined_dim_user", combined_dim_user)
con.register("combined_dim_brand", combined_dim_brand)
con.register("combined_dim_product", combined_dim_product)
con.register("combined_dim_category", combined_dim_category)

print("All combined tables registered in DuckDB")

In [ ]:
query = """
SELECT 'combined_session_summary' AS table_name, COUNT(*) AS row_count FROM combined_session_summary
UNION ALL
SELECT 'combined_dim_user', COUNT(*) FROM combined_dim_user
UNION ALL
SELECT 'combined_dim_brand', COUNT(*) FROM combined_dim_brand
UNION ALL
SELECT 'combined_dim_product', COUNT(*) FROM combined_dim_product
UNION ALL
SELECT 'combined_dim_category', COUNT(*) FROM combined_dim_category;
"""

con.execute(query).df()

In [ ]:
query = """
SELECT
    month,
    COUNT(*) AS row_count
FROM combined_session_summary
GROUP BY month
ORDER BY month;
"""

con.execute(query).df()

In [ ]:
query = """
SELECT
    month,

    COUNT(*) AS total_sessions,
    COUNT(DISTINCT user_id) AS total_users,

    SUM(viewed) AS view_sessions,
    SUM(carted) AS cart_sessions,
    SUM(purchased) AS purchase_sessions,

    ROUND(SUM(revenue), 2) AS total_revenue,

    ROUND(SUM(carted) * 100.0 / NULLIF(SUM(viewed), 0), 2) AS view_to_cart_pct,

    ROUND(SUM(purchased) * 100.0 / NULLIF(SUM(carted), 0), 2) AS cart_to_purchase_pct,

    ROUND(SUM(purchased) * 100.0 / NULLIF(SUM(viewed), 0), 2) AS conversion_rate,

    ROUND((SUM(carted) - SUM(purchased)) * 100.0 / NULLIF(SUM(carted), 0), 2) AS cart_abandonment_pct,

    ROUND(SUM(revenue) / NULLIF(COUNT(*), 0), 2) AS revenue_per_session

FROM combined_session_summary
GROUP BY month
ORDER BY month;
"""

executive_kpis = con.execute(query).df()
executive_kpis

In [ ]:
executive_kpis.to_csv("/kaggle/working/bi_executive_kpis.csv", index=False)

print("Saved: bi_executive_kpis.csv")

In [ ]:
query = """
SELECT
    month,
    '1. Product Views' AS funnel_stage,
    SUM(viewed) AS sessions
FROM combined_session_summary
GROUP BY month

UNION ALL

SELECT
    month,
    '2. Add to Cart' AS funnel_stage,
    SUM(carted) AS sessions
FROM combined_session_summary
GROUP BY month

UNION ALL

SELECT
    month,
    '3. Purchase' AS funnel_stage,
    SUM(purchased) AS sessions
FROM combined_session_summary
GROUP BY month

ORDER BY month, funnel_stage;
"""

bi_discovery_funnel = con.execute(query).df()
bi_discovery_funnel

In [ ]:
bi_discovery_funnel.to_csv("/kaggle/working/bi_discovery_funnel.csv", index=False)

print("Saved: bi_discovery_funnel.csv")

In [ ]:
query = """
SELECT
    month,

    SUM(viewed) AS view_sessions,
    SUM(carted) AS cart_sessions,
    SUM(purchased) AS purchase_sessions,

    SUM(viewed) - SUM(carted) AS view_to_cart_dropoff,
    SUM(carted) - SUM(purchased) AS cart_to_purchase_dropoff,

    ROUND((SUM(viewed) - SUM(carted)) * 100.0 / NULLIF(SUM(viewed), 0), 2) AS view_to_cart_dropoff_pct,

    ROUND((SUM(carted) - SUM(purchased)) * 100.0 / NULLIF(SUM(carted), 0), 2) AS cart_to_purchase_dropoff_pct,

    ROUND(SUM(carted) * 100.0 / NULLIF(SUM(viewed), 0), 2) AS view_to_cart_pct,

    ROUND(SUM(purchased) * 100.0 / NULLIF(SUM(carted), 0), 2) AS cart_to_purchase_pct,

    ROUND(SUM(purchased) * 100.0 / NULLIF(SUM(viewed), 0), 2) AS view_to_purchase_pct

FROM combined_session_summary
GROUP BY month
ORDER BY month;
"""

bi_funnel_rates = con.execute(query).df()
bi_funnel_rates

In [ ]:
bi_funnel_rates.to_csv("/kaggle/working/bi_funnel_rates.csv", index=False)

print("Saved: bi_funnel_rates.csv")

In [ ]:
query = """
SELECT
    date,
    month,
    week_number,
    day_of_week,

    COUNT(*) AS total_sessions,
    COUNT(DISTINCT user_id) AS total_users,

    SUM(viewed) AS view_sessions,
    SUM(carted) AS cart_sessions,
    SUM(purchased) AS purchase_sessions,

    ROUND(SUM(revenue), 2) AS total_revenue,

    ROUND(SUM(purchased) * 100.0 / NULLIF(SUM(viewed), 0), 2) AS conversion_rate,

    ROUND((SUM(carted) - SUM(purchased)) * 100.0 / NULLIF(SUM(carted), 0), 2) AS cart_abandonment_pct,

    ROUND(SUM(revenue) / NULLIF(COUNT(*), 0), 2) AS revenue_per_session

FROM combined_session_summary
GROUP BY
    date,
    month,
    week_number,
    day_of_week
ORDER BY
    date;
"""

bi_daily_trend = con.execute(query).df()
bi_daily_trend

In [ ]:
bi_daily_trend.to_csv("/kaggle/working/bi_daily_trend.csv", index=False)

print("Saved: bi_daily_trend.csv")

In [ ]:
query = """
SELECT
    month,
    day_of_week,

    COUNT(*) AS total_sessions,
    COUNT(DISTINCT user_id) AS total_users,

    SUM(viewed) AS view_sessions,
    SUM(carted) AS cart_sessions,
    SUM(purchased) AS purchase_sessions,

    ROUND(SUM(revenue), 2) AS total_revenue,

    ROUND(SUM(purchased) * 100.0 / NULLIF(SUM(viewed), 0), 2) AS conversion_rate,

    ROUND(SUM(carted) * 100.0 / NULLIF(SUM(viewed), 0), 2) AS view_to_cart_pct,

    ROUND(SUM(purchased) * 100.0 / NULLIF(SUM(carted), 0), 2) AS cart_to_purchase_pct,

    ROUND(SUM(revenue) / NULLIF(COUNT(*), 0), 2) AS revenue_per_session

FROM combined_session_summary
GROUP BY
    month,
    day_of_week
ORDER BY
    month,
    total_revenue DESC;
"""

bi_day_of_week_performance = con.execute(query).df()
bi_day_of_week_performance

In [ ]:
bi_day_of_week_performance.to_csv("/kaggle/working/bi_day_of_week_performance.csv", index=False)

print("Saved: bi_day_of_week_performance.csv")

In [ ]:
query = """
SELECT
    month,
    category_l1,

    SUM(view_sessions) AS view_sessions,
    SUM(cart_sessions) AS cart_sessions,
    SUM(purchase_sessions) AS purchase_sessions,

    ROUND(SUM(revenue), 2) AS total_revenue,

    ROUND(SUM(cart_sessions) * 100.0 / NULLIF(SUM(view_sessions), 0), 2) AS view_to_cart_pct,

    ROUND(SUM(purchase_sessions) * 100.0 / NULLIF(SUM(cart_sessions), 0), 2) AS cart_to_purchase_pct,

    ROUND(SUM(purchase_sessions) * 100.0 / NULLIF(SUM(view_sessions), 0), 2) AS conversion_rate,

    ROUND((SUM(cart_sessions) - SUM(purchase_sessions)) * 100.0 / NULLIF(SUM(cart_sessions), 0), 2) AS cart_abandonment_pct,

    ROUND(SUM(revenue) / NULLIF(SUM(view_sessions), 0), 2) AS revenue_per_view

FROM combined_dim_category
GROUP BY
    month,
    category_l1
ORDER BY
    month,
    total_revenue DESC;
"""

bi_category_performance = con.execute(query).df()
bi_category_performance

In [ ]:
bi_category_performance.to_csv("/kaggle/working/bi_category_performance.csv", index=False)

print("Saved: bi_category_performance.csv")

In [ ]:
query = """
SELECT
    month,
    brand,

    SUM(view_sessions) AS view_sessions,
    SUM(cart_sessions) AS cart_sessions,
    SUM(purchase_sessions) AS purchase_sessions,

    ROUND(SUM(revenue), 2) AS total_revenue,

    ROUND(SUM(cart_sessions) * 100.0 / NULLIF(SUM(view_sessions), 0), 2) AS view_to_cart_pct,

    ROUND(SUM(purchase_sessions) * 100.0 / NULLIF(SUM(cart_sessions), 0), 2) AS cart_to_purchase_pct,

    ROUND(SUM(purchase_sessions) * 100.0 / NULLIF(SUM(view_sessions), 0), 2) AS conversion_rate,

    ROUND((SUM(cart_sessions) - SUM(purchase_sessions)) * 100.0 / NULLIF(SUM(cart_sessions), 0), 2) AS cart_abandonment_pct,

    ROUND(SUM(revenue) / NULLIF(SUM(view_sessions), 0), 2) AS revenue_per_view

FROM combined_dim_brand
WHERE brand IS NOT NULL
GROUP BY
    month,
    brand
ORDER BY
    month,
    total_revenue DESC;
"""

bi_brand_performance = con.execute(query).df()
bi_brand_performance

In [ ]:
bi_brand_performance = bi_brand_performance.fillna(0)

bi_brand_performance.to_csv("/kaggle/working/bi_brand_performance.csv", index=False)

print("Saved: bi_brand_performance.csv")

In [ ]:
query = """
SELECT
    month,
    product_id,
    brand,
    category_l1,
    category_l2,
    price_tier,

    ROUND(AVG(avg_price), 2) AS avg_price,

    SUM(total_views) AS total_views,
    SUM(total_carts) AS total_carts,
    SUM(total_purchases) AS total_purchases,

    ROUND(SUM(total_carts) * 100.0 / NULLIF(SUM(total_views), 0), 2) AS cart_rate,

    ROUND(SUM(total_purchases) * 100.0 / NULLIF(SUM(total_views), 0), 2) AS conversion_rate,

    ROUND(SUM(total_purchases) * 100.0 / NULLIF(SUM(total_carts), 0), 2) AS cart_to_purchase_pct

FROM combined_dim_product
GROUP BY
    month,
    product_id,
    brand,
    category_l1,
    category_l2,
    price_tier
ORDER BY
    month,
    total_purchases DESC;
"""

bi_product_ranking = con.execute(query).df()
bi_product_ranking

In [ ]:
bi_product_ranking = bi_product_ranking.fillna(0)

bi_product_ranking.to_csv("/kaggle/working/bi_product_ranking.csv", index=False)

print("Saved: bi_product_ranking.csv")

In [ ]:
query = """
WITH product_stats AS (
    SELECT
        month,
        product_id,
        brand,
        category_l1,
        category_l2,
        price_tier,

        ROUND(AVG(avg_price), 2) AS avg_price,

        SUM(total_views) AS total_views,
        SUM(total_carts) AS total_carts,
        SUM(total_purchases) AS total_purchases,

        ROUND(SUM(total_carts) * 100.0 / NULLIF(SUM(total_views), 0), 2) AS cart_rate,

        ROUND(SUM(total_purchases) * 100.0 / NULLIF(SUM(total_views), 0), 2) AS conversion_rate

    FROM combined_dim_product
    GROUP BY
        month,
        product_id,
        brand,
        category_l1,
        category_l2,
        price_tier
),

thresholds AS (
    SELECT
        month,
        AVG(total_views) AS avg_views,
        AVG(conversion_rate) AS avg_conversion_rate
    FROM product_stats
    GROUP BY month
)

SELECT
    p.month,
    p.product_id,
    p.brand,
    p.category_l1,
    p.category_l2,
    p.price_tier,
    p.avg_price,
    p.total_views,
    p.total_carts,
    p.total_purchases,
    p.cart_rate,
    p.conversion_rate,

    ROUND(t.avg_views, 2) AS month_avg_views,
    ROUND(t.avg_conversion_rate, 2) AS month_avg_conversion_rate,

    CASE
        WHEN p.total_views >= t.avg_views
             AND p.conversion_rate >= t.avg_conversion_rate
        THEN 'Winner Product'

        WHEN p.total_views < t.avg_views
             AND p.conversion_rate >= t.avg_conversion_rate
        THEN 'Hidden Gem - Promote Higher'

        WHEN p.total_views >= t.avg_views
             AND p.conversion_rate < t.avg_conversion_rate
        THEN 'High Visibility Low Conversion - Investigate'

        ELSE 'Low Priority Product'
    END AS product_ranking_segment

FROM product_stats p
JOIN thresholds t
    ON p.month = t.month
ORDER BY
    p.month,
    p.total_purchases DESC;
"""

bi_product_ranking_classification = con.execute(query).df()
bi_product_ranking_classification

In [ ]:
bi_product_ranking_classification = bi_product_ranking_classification.fillna(0)

bi_product_ranking_classification.to_csv(
    "/kaggle/working/bi_product_ranking_classification.csv",
    index=False
)

print("Saved: bi_product_ranking_classification.csv")

In [ ]:
query = """
WITH product_stats AS (
    SELECT
        month,
        product_id,
        brand,
        category_l1,
        category_l2,
        price_tier,

        ROUND(AVG(avg_price), 2) AS avg_price,

        SUM(total_views) AS total_views,
        SUM(total_carts) AS total_carts,
        SUM(total_purchases) AS total_purchases,

        ROUND(SUM(total_carts) * 100.0 / NULLIF(SUM(total_views), 0), 2) AS cart_rate,

        ROUND(SUM(total_purchases) * 100.0 / NULLIF(SUM(total_views), 0), 2) AS conversion_rate

    FROM combined_dim_product
    GROUP BY
        month,
        product_id,
        brand,
        category_l1,
        category_l2,
        price_tier
),

thresholds AS (
    SELECT
        month,
        quantile_cont(total_views, 0.50) AS median_views,
        quantile_cont(total_views, 0.75) AS p75_views,
        quantile_cont(conversion_rate, 0.75) AS p75_conversion_rate
    FROM product_stats
    GROUP BY month
)

SELECT
    p.month,
    p.product_id,
    p.brand,
    p.category_l1,
    p.category_l2,
    p.price_tier,
    p.avg_price,

    p.total_views,
    p.total_carts,
    p.total_purchases,
    p.cart_rate,
    p.conversion_rate,

    ROUND(t.median_views, 2) AS median_views,
    ROUND(t.p75_views, 2) AS p75_views,
    ROUND(t.p75_conversion_rate, 2) AS p75_conversion_rate,

    'Strong Hidden Gem - Promote Higher' AS recommendation

FROM product_stats p
JOIN thresholds t
    ON p.month = t.month

WHERE
    p.total_views < t.p75_views
    AND p.conversion_rate >= t.p75_conversion_rate
    AND p.total_purchases >= 5

ORDER BY
    p.month,
    p.conversion_rate DESC,
    p.total_purchases DESC;
"""

bi_strong_hidden_gems = con.execute(query).df()
bi_strong_hidden_gems

In [ ]:
query = """
WITH product_stats AS (
    SELECT
        month,
        product_id,
        SUM(total_views) AS total_views,
        SUM(total_purchases) AS total_purchases,
        ROUND(SUM(total_purchases) * 100.0 / NULLIF(SUM(total_views), 0), 2) AS conversion_rate
    FROM combined_dim_product
    GROUP BY
        month,
        product_id
)

SELECT
    month,

    COUNT(*) AS total_products,

    ROUND(AVG(total_views), 2) AS avg_views,
    ROUND(quantile_cont(total_views, 0.50), 2) AS median_views,
    ROUND(quantile_cont(total_views, 0.75), 2) AS p75_views,
    ROUND(quantile_cont(total_views, 0.90), 2) AS p90_views,

    ROUND(AVG(conversion_rate), 2) AS avg_conversion_rate,
    ROUND(quantile_cont(conversion_rate, 0.75), 2) AS p75_conversion_rate,
    ROUND(quantile_cont(conversion_rate, 0.90), 2) AS p90_conversion_rate,

    SUM(CASE WHEN total_purchases >= 1 THEN 1 ELSE 0 END) AS products_with_purchase,
    SUM(CASE WHEN total_purchases >= 5 THEN 1 ELSE 0 END) AS products_with_5plus_purchases

FROM product_stats
GROUP BY month
ORDER BY month;
"""

product_threshold_check = con.execute(query).df()
product_threshold_check

In [ ]:
query = """
WITH product_stats AS (
    SELECT
        month,
        product_id,
        brand,
        category_l1,
        category_l2,
        price_tier,

        ROUND(AVG(avg_price), 2) AS avg_price,

        SUM(total_views) AS total_views,
        SUM(total_carts) AS total_carts,
        SUM(total_purchases) AS total_purchases,

        ROUND(SUM(total_carts) * 100.0 / NULLIF(SUM(total_views), 0), 2) AS cart_rate,

        ROUND(SUM(total_purchases) * 100.0 / NULLIF(SUM(total_views), 0), 2) AS conversion_rate

    FROM combined_dim_product
    GROUP BY
        month,
        product_id,
        brand,
        category_l1,
        category_l2,
        price_tier
)

SELECT
    month,
    product_id,
    brand,
    category_l1,
    category_l2,
    price_tier,
    avg_price,
    total_views,
    total_carts,
    total_purchases,
    cart_rate,
    conversion_rate,

    'Hidden Gem - Promote Higher' AS recommendation

FROM product_stats
WHERE
    total_views BETWEEN 20 AND 200
    AND conversion_rate >= 3
    AND total_purchases >= 3

ORDER BY
    month,
    conversion_rate DESC,
    total_purchases DESC;
"""

bi_hidden_gems = con.execute(query).df()
bi_hidden_gems

In [ ]:
bi_hidden_gems.to_csv("/kaggle/working/bi_hidden_gems.csv", index=False)

print("Saved: bi_hidden_gems.csv")

In [ ]:
query = """
WITH product_stats AS (
    SELECT
        month,
        product_id,
        brand,
        category_l1,
        category_l2,
        price_tier,

        ROUND(AVG(avg_price), 2) AS avg_price,

        SUM(total_views) AS total_views,
        SUM(total_carts) AS total_carts,
        SUM(total_purchases) AS total_purchases,

        ROUND(SUM(total_carts) * 100.0 / NULLIF(SUM(total_views), 0), 2) AS cart_rate,

        ROUND(SUM(total_purchases) * 100.0 / NULLIF(SUM(total_views), 0), 2) AS conversion_rate

    FROM combined_dim_product
    GROUP BY
        month,
        product_id,
        brand,
        category_l1,
        category_l2,
        price_tier
)

SELECT
    month,
    product_id,
    brand,
    category_l1,
    category_l2,
    price_tier,
    avg_price,

    total_views,
    total_carts,
    total_purchases,
    cart_rate,
    conversion_rate,

    'High Visibility Low Conversion - Investigate' AS recommendation

FROM product_stats
WHERE
    total_views >= 1000
    AND conversion_rate < 1
    AND total_purchases < 10

ORDER BY
    month,
    total_views DESC;
"""

bi_high_visibility_low_conversion = con.execute(query).df()
bi_high_visibility_low_conversion

In [ ]:
bi_high_visibility_low_conversion.to_csv(
    "/kaggle/working/bi_high_visibility_low_conversion.csv",
    index=False
)

print("Saved: bi_high_visibility_low_conversion.csv")

In [ ]:
query = """
SELECT
    month,
    segment,

    COUNT(DISTINCT user_id) AS total_users,

    SUM(total_sessions) AS total_sessions,
    SUM(total_events) AS total_events,
    SUM(total_views) AS total_views,
    SUM(total_carts) AS total_carts,
    SUM(total_purchases) AS total_purchases,

    ROUND(SUM(total_spend), 2) AS total_revenue,

    ROUND(SUM(total_spend) / NULLIF(COUNT(DISTINCT user_id), 0), 2) AS revenue_per_user,

    ROUND(SUM(total_sessions) * 1.0 / NULLIF(COUNT(DISTINCT user_id), 0), 2) AS sessions_per_user,

    ROUND(SUM(total_purchases) * 1.0 / NULLIF(COUNT(DISTINCT user_id), 0), 2) AS purchases_per_user,

    ROUND(SUM(total_purchases) * 100.0 / NULLIF(SUM(total_views), 0), 2) AS conversion_rate,

    ROUND((SUM(total_carts) - SUM(total_purchases)) * 100.0 / NULLIF(SUM(total_carts), 0), 2) AS cart_abandonment_pct

FROM combined_dim_user
GROUP BY
    month,
    segment
ORDER BY
    month,
    total_revenue DESC;
"""

bi_user_segment_performance = con.execute(query).df()
bi_user_segment_performance

In [ ]:
bi_user_segment_performance = bi_user_segment_performance.fillna(0)

bi_user_segment_performance.to_csv(
    "/kaggle/working/bi_user_segment_performance.csv",
    index=False
)

print("Saved: bi_user_segment_performance.csv")

In [ ]:
query = """
SELECT
    month,
    user_id,
    segment,

    total_sessions,
    total_events,
    total_purchases,
    ROUND(total_spend, 2) AS total_spend,

    ROUND(total_spend / NULLIF(total_sessions, 0), 2) AS spend_per_session,

    ROUND(total_purchases * 1.0 / NULLIF(total_sessions, 0), 2) AS purchases_per_session

FROM combined_dim_user
WHERE total_spend > 0
ORDER BY
    total_spend DESC
LIMIT 1000;
"""

bi_high_value_users = con.execute(query).df()
bi_high_value_users

In [ ]:
bi_high_value_users.to_csv(
    "/kaggle/working/bi_high_value_users.csv",
    index=False
)

print("Saved: bi_high_value_users.csv")

In [ ]:
query = """
WITH monthly AS (
    SELECT
        month,

        COUNT(*) AS total_sessions,
        COUNT(DISTINCT user_id) AS total_users,

        SUM(viewed) AS view_sessions,
        SUM(carted) AS cart_sessions,
        SUM(purchased) AS purchase_sessions,

        ROUND(SUM(revenue), 2) AS total_revenue,

        ROUND(SUM(carted) * 100.0 / NULLIF(SUM(viewed), 0), 2) AS view_to_cart_pct,

        ROUND(SUM(purchased) * 100.0 / NULLIF(SUM(carted), 0), 2) AS cart_to_purchase_pct,

        ROUND(SUM(purchased) * 100.0 / NULLIF(SUM(viewed), 0), 2) AS conversion_rate,

        ROUND((SUM(carted) - SUM(purchased)) * 100.0 / NULLIF(SUM(carted), 0), 2) AS cart_abandonment_pct,

        ROUND(SUM(revenue) / NULLIF(COUNT(*), 0), 2) AS revenue_per_session

    FROM combined_session_summary
    GROUP BY month
),

pivoted AS (
    SELECT
        MAX(CASE WHEN month = 'October' THEN total_sessions END) AS oct_sessions,
        MAX(CASE WHEN month = 'November' THEN total_sessions END) AS nov_sessions,

        MAX(CASE WHEN month = 'October' THEN total_users END) AS oct_users,
        MAX(CASE WHEN month = 'November' THEN total_users END) AS nov_users,

        MAX(CASE WHEN month = 'October' THEN view_sessions END) AS oct_view_sessions,
        MAX(CASE WHEN month = 'November' THEN view_sessions END) AS nov_view_sessions,

        MAX(CASE WHEN month = 'October' THEN cart_sessions END) AS oct_cart_sessions,
        MAX(CASE WHEN month = 'November' THEN cart_sessions END) AS nov_cart_sessions,

        MAX(CASE WHEN month = 'October' THEN purchase_sessions END) AS oct_purchase_sessions,
        MAX(CASE WHEN month = 'November' THEN purchase_sessions END) AS nov_purchase_sessions,

        MAX(CASE WHEN month = 'October' THEN total_revenue END) AS oct_revenue,
        MAX(CASE WHEN month = 'November' THEN total_revenue END) AS nov_revenue,

        MAX(CASE WHEN month = 'October' THEN conversion_rate END) AS oct_conversion_rate,
        MAX(CASE WHEN month = 'November' THEN conversion_rate END) AS nov_conversion_rate,

        MAX(CASE WHEN month = 'October' THEN cart_abandonment_pct END) AS oct_cart_abandonment_pct,
        MAX(CASE WHEN month = 'November' THEN cart_abandonment_pct END) AS nov_cart_abandonment_pct,

        MAX(CASE WHEN month = 'October' THEN revenue_per_session END) AS oct_revenue_per_session,
        MAX(CASE WHEN month = 'November' THEN revenue_per_session END) AS nov_revenue_per_session
    FROM monthly
)

SELECT
    oct_sessions,
    nov_sessions,
    nov_sessions - oct_sessions AS session_change,
    ROUND((nov_sessions - oct_sessions) * 100.0 / NULLIF(oct_sessions, 0), 2) AS session_change_pct,

    oct_users,
    nov_users,
    nov_users - oct_users AS user_change,
    ROUND((nov_users - oct_users) * 100.0 / NULLIF(oct_users, 0), 2) AS user_change_pct,

    oct_view_sessions,
    nov_view_sessions,
    nov_view_sessions - oct_view_sessions AS view_session_change,
    ROUND((nov_view_sessions - oct_view_sessions) * 100.0 / NULLIF(oct_view_sessions, 0), 2) AS view_session_change_pct,

    oct_cart_sessions,
    nov_cart_sessions,
    nov_cart_sessions - oct_cart_sessions AS cart_session_change,
    ROUND((nov_cart_sessions - oct_cart_sessions) * 100.0 / NULLIF(oct_cart_sessions, 0), 2) AS cart_session_change_pct,

    oct_purchase_sessions,
    nov_purchase_sessions,
    nov_purchase_sessions - oct_purchase_sessions AS purchase_session_change,
    ROUND((nov_purchase_sessions - oct_purchase_sessions) * 100.0 / NULLIF(oct_purchase_sessions, 0), 2) AS purchase_session_change_pct,

    oct_revenue,
    nov_revenue,
    ROUND(nov_revenue - oct_revenue, 2) AS revenue_change,
    ROUND((nov_revenue - oct_revenue) * 100.0 / NULLIF(oct_revenue, 0), 2) AS revenue_change_pct,

    oct_conversion_rate,
    nov_conversion_rate,
    ROUND(nov_conversion_rate - oct_conversion_rate, 2) AS conversion_rate_change,

    oct_cart_abandonment_pct,
    nov_cart_abandonment_pct,
    ROUND(nov_cart_abandonment_pct - oct_cart_abandonment_pct, 2) AS cart_abandonment_change,

    oct_revenue_per_session,
    nov_revenue_per_session,
    ROUND(nov_revenue_per_session - oct_revenue_per_session, 2) AS revenue_per_session_change

FROM pivoted;
"""

bi_mom_kpi_change = con.execute(query).df()
bi_mom_kpi_change

In [ ]:
bi_mom_kpi_change.to_csv(
    "/kaggle/working/bi_mom_kpi_change.csv",
    index=False
)

print("Saved: bi_mom_kpi_change.csv")

In [ ]:
query = """
WITH category_monthly AS (
    SELECT
        month,
        category_l1,

        SUM(view_sessions) AS view_sessions,
        SUM(cart_sessions) AS cart_sessions,
        SUM(purchase_sessions) AS purchase_sessions,

        ROUND(SUM(revenue), 2) AS total_revenue,

        ROUND(SUM(purchase_sessions) * 100.0 / NULLIF(SUM(view_sessions), 0), 2) AS conversion_rate,

        ROUND((SUM(cart_sessions) - SUM(purchase_sessions)) * 100.0 / NULLIF(SUM(cart_sessions), 0), 2) AS cart_abandonment_pct,

        ROUND(SUM(revenue) / NULLIF(SUM(view_sessions), 0), 2) AS revenue_per_view

    FROM combined_dim_category
    GROUP BY
        month,
        category_l1
),

pivoted AS (
    SELECT
        category_l1,

        MAX(CASE WHEN month = 'October' THEN view_sessions END) AS oct_view_sessions,
        MAX(CASE WHEN month = 'November' THEN view_sessions END) AS nov_view_sessions,

        MAX(CASE WHEN month = 'October' THEN purchase_sessions END) AS oct_purchase_sessions,
        MAX(CASE WHEN month = 'November' THEN purchase_sessions END) AS nov_purchase_sessions,

        MAX(CASE WHEN month = 'October' THEN total_revenue END) AS oct_revenue,
        MAX(CASE WHEN month = 'November' THEN total_revenue END) AS nov_revenue,

        MAX(CASE WHEN month = 'October' THEN conversion_rate END) AS oct_conversion_rate,
        MAX(CASE WHEN month = 'November' THEN conversion_rate END) AS nov_conversion_rate,

        MAX(CASE WHEN month = 'October' THEN cart_abandonment_pct END) AS oct_cart_abandonment_pct,
        MAX(CASE WHEN month = 'November' THEN cart_abandonment_pct END) AS nov_cart_abandonment_pct,

        MAX(CASE WHEN month = 'October' THEN revenue_per_view END) AS oct_revenue_per_view,
        MAX(CASE WHEN month = 'November' THEN revenue_per_view END) AS nov_revenue_per_view

    FROM category_monthly
    GROUP BY category_l1
)

SELECT
    category_l1,

    COALESCE(oct_view_sessions, 0) AS oct_view_sessions,
    COALESCE(nov_view_sessions, 0) AS nov_view_sessions,
    COALESCE(nov_view_sessions, 0) - COALESCE(oct_view_sessions, 0) AS view_session_change,

    COALESCE(oct_purchase_sessions, 0) AS oct_purchase_sessions,
    COALESCE(nov_purchase_sessions, 0) AS nov_purchase_sessions,
    COALESCE(nov_purchase_sessions, 0) - COALESCE(oct_purchase_sessions, 0) AS purchase_session_change,

    COALESCE(oct_revenue, 0) AS oct_revenue,
    COALESCE(nov_revenue, 0) AS nov_revenue,
    ROUND(COALESCE(nov_revenue, 0) - COALESCE(oct_revenue, 0), 2) AS revenue_change,

    ROUND(
        (COALESCE(nov_revenue, 0) - COALESCE(oct_revenue, 0)) * 100.0 
        / NULLIF(oct_revenue, 0),
        2
    ) AS revenue_change_pct,

    COALESCE(oct_conversion_rate, 0) AS oct_conversion_rate,
    COALESCE(nov_conversion_rate, 0) AS nov_conversion_rate,
    ROUND(COALESCE(nov_conversion_rate, 0) - COALESCE(oct_conversion_rate, 0), 2) AS conversion_rate_change,

    COALESCE(oct_cart_abandonment_pct, 0) AS oct_cart_abandonment_pct,
    COALESCE(nov_cart_abandonment_pct, 0) AS nov_cart_abandonment_pct,
    ROUND(COALESCE(nov_cart_abandonment_pct, 0) - COALESCE(oct_cart_abandonment_pct, 0), 2) AS cart_abandonment_change,

    COALESCE(oct_revenue_per_view, 0) AS oct_revenue_per_view,
    COALESCE(nov_revenue_per_view, 0) AS nov_revenue_per_view,
    ROUND(COALESCE(nov_revenue_per_view, 0) - COALESCE(oct_revenue_per_view, 0), 2) AS revenue_per_view_change

FROM pivoted
ORDER BY revenue_change ASC;
"""

bi_category_mom_change = con.execute(query).df()
bi_category_mom_change

In [ ]:
bi_category_mom_change = bi_category_mom_change.fillna(0)

bi_category_mom_change.to_csv(
    "/kaggle/working/bi_category_mom_change.csv",
    index=False
)

print("Saved: bi_category_mom_change.csv")

In [ ]:
query = """
WITH brand_monthly AS (
    SELECT
        month,
        brand,

        SUM(view_sessions) AS view_sessions,
        SUM(cart_sessions) AS cart_sessions,
        SUM(purchase_sessions) AS purchase_sessions,

        ROUND(SUM(revenue), 2) AS total_revenue,

        ROUND(SUM(purchase_sessions) * 100.0 / NULLIF(SUM(view_sessions), 0), 2) AS conversion_rate,

        ROUND((SUM(cart_sessions) - SUM(purchase_sessions)) * 100.0 / NULLIF(SUM(cart_sessions), 0), 2) AS cart_abandonment_pct,

        ROUND(SUM(revenue) / NULLIF(SUM(view_sessions), 0), 2) AS revenue_per_view

    FROM combined_dim_brand
    WHERE brand IS NOT NULL
    GROUP BY
        month,
        brand
),

pivoted AS (
    SELECT
        brand,

        MAX(CASE WHEN month = 'October' THEN view_sessions END) AS oct_view_sessions,
        MAX(CASE WHEN month = 'November' THEN view_sessions END) AS nov_view_sessions,

        MAX(CASE WHEN month = 'October' THEN purchase_sessions END) AS oct_purchase_sessions,
        MAX(CASE WHEN month = 'November' THEN purchase_sessions END) AS nov_purchase_sessions,

        MAX(CASE WHEN month = 'October' THEN total_revenue END) AS oct_revenue,
        MAX(CASE WHEN month = 'November' THEN total_revenue END) AS nov_revenue,

        MAX(CASE WHEN month = 'October' THEN conversion_rate END) AS oct_conversion_rate,
        MAX(CASE WHEN month = 'November' THEN conversion_rate END) AS nov_conversion_rate,

        MAX(CASE WHEN month = 'October' THEN cart_abandonment_pct END) AS oct_cart_abandonment_pct,
        MAX(CASE WHEN month = 'November' THEN cart_abandonment_pct END) AS nov_cart_abandonment_pct,

        MAX(CASE WHEN month = 'October' THEN revenue_per_view END) AS oct_revenue_per_view,
        MAX(CASE WHEN month = 'November' THEN revenue_per_view END) AS nov_revenue_per_view

    FROM brand_monthly
    GROUP BY brand
)

SELECT
    brand,

    COALESCE(oct_view_sessions, 0) AS oct_view_sessions,
    COALESCE(nov_view_sessions, 0) AS nov_view_sessions,
    COALESCE(nov_view_sessions, 0) - COALESCE(oct_view_sessions, 0) AS view_session_change,

    COALESCE(oct_purchase_sessions, 0) AS oct_purchase_sessions,
    COALESCE(nov_purchase_sessions, 0) AS nov_purchase_sessions,
    COALESCE(nov_purchase_sessions, 0) - COALESCE(oct_purchase_sessions, 0) AS purchase_session_change,

    COALESCE(oct_revenue, 0) AS oct_revenue,
    COALESCE(nov_revenue, 0) AS nov_revenue,
    ROUND(COALESCE(nov_revenue, 0) - COALESCE(oct_revenue, 0), 2) AS revenue_change,

    ROUND(
        (COALESCE(nov_revenue, 0) - COALESCE(oct_revenue, 0)) * 100.0 
        / NULLIF(oct_revenue, 0),
        2
    ) AS revenue_change_pct,

    COALESCE(oct_conversion_rate, 0) AS oct_conversion_rate,
    COALESCE(nov_conversion_rate, 0) AS nov_conversion_rate,
    ROUND(COALESCE(nov_conversion_rate, 0) - COALESCE(oct_conversion_rate, 0), 2) AS conversion_rate_change,

    COALESCE(oct_cart_abandonment_pct, 0) AS oct_cart_abandonment_pct,
    COALESCE(nov_cart_abandonment_pct, 0) AS nov_cart_abandonment_pct,
    ROUND(COALESCE(nov_cart_abandonment_pct, 0) - COALESCE(oct_cart_abandonment_pct, 0), 2) AS cart_abandonment_change,

    COALESCE(oct_revenue_per_view, 0) AS oct_revenue_per_view,
    COALESCE(nov_revenue_per_view, 0) AS nov_revenue_per_view,
    ROUND(COALESCE(nov_revenue_per_view, 0) - COALESCE(oct_revenue_per_view, 0), 2) AS revenue_per_view_change

FROM pivoted
ORDER BY revenue_change ASC;
"""

bi_brand_mom_change = con.execute(query).df()
bi_brand_mom_change

In [ ]:
bi_brand_mom_change = bi_brand_mom_change.fillna(0)

bi_brand_mom_change.to_csv(
    "/kaggle/working/bi_brand_mom_change.csv",
    index=False
)

print("Saved: bi_brand_mom_change.csv")

In [ ]:
import os

exported_files = []

for file in os.listdir("/kaggle/working"):
    if file.endswith(".csv"):
        exported_files.append(file)

exported_files

In [ ]:
import os

zip_path = "/kaggle/working/ecommerce_bi_ready_files.zip"

if os.path.exists(zip_path):
    os.remove(zip_path)
    print("Deleted large zip file")
else:
    print("Zip file not found")

In [ ]:
import os
import shutil

output_folder = "/kaggle/working/bi_exports_only"

os.makedirs(output_folder, exist_ok=True)

bi_files = [
    "bi_category_performance.csv",
    "bi_day_of_week_performance.csv",
    "bi_funnel_rates.csv",
    "bi_product_ranking_classification.csv",
    "bi_category_mom_change.csv",
    "bi_executive_kpis.csv",
    "bi_brand_mom_change.csv",
    "bi_high_visibility_low_conversion.csv",
    "bi_discovery_funnel.csv",
    "bi_brand_performance.csv",
    "bi_mom_kpi_change.csv",
    "bi_high_value_users.csv",
    "bi_daily_trend.csv",
    "bi_product_ranking.csv",
    "bi_hidden_gems.csv",
    "bi_user_segment_performance.csv"
]

for file in bi_files:
    src = f"/kaggle/working/{file}"
    dst = f"{output_folder}/{file}"
    if os.path.exists(src):
        shutil.copy(src, dst)

print("Copied BI files only")

In [ ]:
shutil.make_archive(
    "/kaggle/working/bi_exports_only",
    "zip",
    output_folder
)

print("Created small zip: bi_exports_only.zip")